# IDH Mutation Prediction in Glioma — Full Pipeline
### Beyond Histological Grading: Radiomics, CNN, and Hybrid Fusion

**Datasets:** UCSF-PDGM (training) · EGD (external validation) · TCGA BraTS 2021 (external validation)

**Tracks:** Track A — 7 sequences (T1, T1c, T2, FLAIR, ADC, FA, MD) · Track B — 4 sequences (T1, T1c, T2, FLAIR)

**Arms:** Radiomics (PyRadiomics + SVM/XGBoost) · CNN (ResNet18) · Hybrid Fusion

---
**Cell index:**
- Cell 1: Setup — install dependencies, verify datasets mounted
- Cell 2: Labels — build patient list, binarize IDH
- Cell 3: Splits — stratified 5-fold, unit test
- Cell 4: Preprocessing — clip, normalize, build CNN tensors + summary
- Cell 5: CNN training — Track A (ResNet18 7-channel)
- Cell 6: CNN training — Track B (ResNet18 4-channel)
- Cell 7: Radiomics feature extraction
- Cell 8: Radiomics feature selection + classification (SVM/XGBoost) + SHAP
- Cell 9: Hybrid fusion — embedding extraction + PCA + concatenation
- Cell 10: Fusion classifier (SVM/XGBoost) — early and late fusion
- Cell 11: External validation — EGD + TCGA (Track B models only)
- Cell 12: Ablation studies — 4-seq vs 5-seq vs 7-seq
- Cell 13: Statistical tests — DeLong, bootstrap CI, Bonferroni
- Cell 14: Figures — ROC curves, SHAP, Grad-CAM, confusion matrices
- Cell 15: Results table + final summary

## Cell 1 — Setup
**Run every new session before anything else.**

In [1]:
!pip install nibabel SimpleITK scikit-image -q

import os, json, glob
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
from skimage.transform import resize
from sklearn.model_selection import StratifiedKFold

WORKING = '/kaggle/working'
os.makedirs(f'{WORKING}/trackA_slices', exist_ok=True)
# Track B derived at runtime from trackA[:, :4, :, :] — no separate directory needed

# Input paths — all 10 batch datasets
KAGGLE_USER  = 'adesaladaniel'
BATCH_ROOTS  = [f'/kaggle/input/datasets/{KAGGLE_USER}/ucsf-pdgm-batch-{i:02d}' for i in range(1, 11)]

# Sequence filenames
TRACK_A_SEQS = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TRACK_B_SEQS = ['T1', 'T1c', 'T2', 'FLAIR']
MASK_SEQS    = ['tumor_segmentation', 'brain_segmentation']

print('Dependencies loaded.')
print(f'Batch roots found: {sum(os.path.exists(b) for b in BATCH_ROOTS)}/10')
for b in BATCH_ROOTS:
    status = 'OK     ' if os.path.exists(b) else 'MISSING'
    print(f'  {status}  {b}')

Dependencies loaded.
Batch roots found: 10/10
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-02
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-03
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-04
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-05
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-06
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-07
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-08
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-09
  OK       /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10


## Cell 2 — Labels
**build patient list, binarize IDH**

In [2]:
import requests

# ── DISCOVER ALL PATIENT FOLDERS ─────────────────────────────────────────────
patient_dirs = {}
for batch_root in BATCH_ROOTS:
    if not os.path.exists(batch_root):
        continue
    for entry in sorted(os.listdir(batch_root)):
        if entry.startswith('UCSF-PDGM-'):
            full_path = os.path.join(batch_root, entry)
            if os.path.isdir(full_path):
                patient_dirs[entry] = full_path

print(f'Patient folders found: {len(patient_dirs)}')
if len(patient_dirs) == 0:
    print('ERROR: No patient folders found.')
    raise SystemExit

# Print first 3 and last 3 to verify structure
sample = sorted(patient_dirs.keys())
for pid in sample[:3] + sample[-3:]:
    print(f'  {pid}  →  {patient_dirs[pid]}')

print()

# ── LOAD METADATA AND BINARIZE IDH ───────────────────────────────────────────
meta_path = f'{WORKING}/meta.csv'
if not os.path.exists(meta_path):
    print('Downloading metadata CSV...')
    r = requests.get('https://www.cancerimagingarchive.net/wp-content/uploads/UCSF-PDGM-metadata.csv')
    with open(meta_path, 'wb') as f:
        f.write(r.content)
    print('Downloaded.')

df_meta = pd.read_csv(meta_path)

# Standardize patient ID to UCSF-PDGM-XXXX
df_meta['ID_padded'] = df_meta['ID'].apply(
    lambda p: f'UCSF-PDGM-{int(str(p).split("-")[-1]):04d}'
)

# Binarize IDH: wildtype=0, all mutation variants=1
def binarize_idh(val):
    if pd.isna(val):
        return None
    v = str(val).strip().lower()
    if v == 'wildtype':
        return 0
    return 1

df_meta['IDH_binary'] = df_meta['IDH'].apply(binarize_idh)

# Exclude follow-up scans
FOLLOWUP_IDS = [
    'UCSF-PDGM-0433', 'UCSF-PDGM-0431', 'UCSF-PDGM-0396',
    'UCSF-PDGM-0429', 'UCSF-PDGM-0409', 'UCSF-PDGM-0391'
]
df_labels = df_meta[
    df_meta['IDH_binary'].notna() &
    ~df_meta['ID_padded'].isin(FOLLOWUP_IDS)
][['ID_padded', 'IDH', 'IDH_binary', 'WHO CNS Grade',
   'Final pathologic diagnosis (WHO 2021)', 'MGMT status']].copy()
df_labels = df_labels.reset_index(drop=True)
df_labels.rename(columns={'ID_padded': 'patient_id'}, inplace=True)

# Keep only patients we have folders for
df_labels = df_labels[df_labels['patient_id'].isin(patient_dirs)].reset_index(drop=True)
df_labels['folder_path'] = df_labels['patient_id'].map(patient_dirs)

wt = (df_labels['IDH_binary'] == 0).sum()
mt = (df_labels['IDH_binary'] == 1).sum()
print(f'Patients with labels + folders : {len(df_labels)}  (expect 495)')
print(f'IDH Wildtype                   : {wt}')
print(f'IDH Mutant                     : {mt}')
print(f'Class ratio                    : {wt/mt:.2f}:1')
print()
print('Grade distribution:')
print(df_labels['WHO CNS Grade'].value_counts().sort_index().to_string())

df_labels.to_csv(f'{WORKING}/labels.csv', index=False)
print(f'\nSaved: {WORKING}/labels.csv')

Patient folders found: 495
  UCSF-PDGM-0004  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0004
  UCSF-PDGM-0005  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0005
  UCSF-PDGM-0007  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-01/UCSF-PDGM-0007
  UCSF-PDGM-0539  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0539
  UCSF-PDGM-0540  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0540
  UCSF-PDGM-0541  →  /kaggle/input/datasets/adesaladaniel/ucsf-pdgm-batch-10/UCSF-PDGM-0541

Downloaded.
Patients with labels + folders : 495  (expect 495)
IDH Wildtype                   : 392
IDH Mutant                     : 103
Class ratio                    : 3.81:1

Grade distribution:
WHO CNS Grade
2     56
3     43
4    396

Saved: /kaggle/working/labels.csv


## Cell 3: Splits 
**stratified 5-fold, unit test**

In [3]:
# ── STRATIFIED 5-FOLD SPLIT ───────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

patient_ids = df_labels['patient_id'].values
idh_labels  = df_labels['IDH_binary'].values.astype(int)
fold_assignments = np.zeros(len(df_labels), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(patient_ids, idh_labels)):
    fold_assignments[val_idx] = fold_idx

df_labels['fold'] = fold_assignments
df_labels.to_csv(f'{WORKING}/labels.csv', index=False)

# Save fold assignments as JSON
splits = {}
for fold in range(5):
    fold_patients = df_labels[df_labels['fold'] == fold]['patient_id'].tolist()
    splits[f'fold_{fold}'] = fold_patients

with open(f'{WORKING}/patient_splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

# ── UNIT TEST: ZERO PATIENT OVERLAP ──────────────────────────────────────────
print('Running data leakage unit test...')
print()
all_passed = True
for fold in range(5):
    val_patients   = set(splits[f'fold_{fold}'])
    train_patients = set()
    for other_fold in range(5):
        if other_fold != fold:
            train_patients.update(splits[f'fold_{other_fold}'])
    overlap = val_patients & train_patients
    if overlap:
        print(f'  FAIL fold {fold}: {len(overlap)} patients in both train and val!')
        all_passed = False
    else:
        val_wt = sum(1 for p in val_patients
                     if df_labels.loc[df_labels['patient_id']==p, 'IDH_binary'].values[0] == 0)
        val_mt = len(val_patients) - val_wt
        print(f'  PASS fold {fold}: {len(val_patients)} patients — {val_wt} wildtype / {val_mt} mutant')

print()
if all_passed:
    print('All folds pass — zero patient ID overlap confirmed.')
    print(f'Saved: {WORKING}/patient_splits.json')
else:
    print('FAILED — do not proceed.')
    raise SystemExit

Running data leakage unit test...

  PASS fold 0: 99 patients — 79 wildtype / 20 mutant
  PASS fold 1: 99 patients — 79 wildtype / 20 mutant
  PASS fold 2: 99 patients — 78 wildtype / 21 mutant
  PASS fold 3: 99 patients — 78 wildtype / 21 mutant
  PASS fold 4: 99 patients — 78 wildtype / 21 mutant

All folds pass — zero patient ID overlap confirmed.
Saved: /kaggle/working/patient_splits.json


## Cell 4: Preprocessing 
**clip, normalize, build CNN tensors**

In [4]:
raise SystemExit("OLD CELL 4 RETIRED — use new Cell 4 (verify mount) instead")
# ── OLD CELL 4 — PREPROCESSING (RETIRED 2026-06-26) ──────────────────────────
# This cell has been retired. Tensors are now permanently stored at:
#   adesaladaniel/glioma-idh-tensors
# Mounted at: /kaggle/input/glioma-idh-tensors/
# Use new Cell 4 (verification cell) instead.
# This code is kept for reference only — do NOT run it.
# ─────────────────────────────────────────────────────────────────────────────

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
TRACK_A_SEQS  = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TUMOR_SEQ     = 'tumor_segmentation'
BRAIN_SEQ     = 'brain_segmentation'
SLICE_SIZE    = 224
TUMOR_THR     = 0.001

# ── RELOAD STATE ──────────────────────────────────────────────────────────────
df_labels     = pd.read_csv(f'{WORKING}/labels.csv')
progress_path = f'{WORKING}/progress.json'

if os.path.exists(progress_path):
    with open(progress_path) as f:
        progress = json.load(f)
    completed_raw = set(progress.get('completed', []))
    completed = set(
        p for p in completed_raw
        if os.path.exists(f'{WORKING}/trackA_slices/{p}.npy')
    )
    lost = completed_raw - completed
    if lost:
        print(f'WARNING: {len(lost)} patients marked done but .npy files missing — will reprocess')
    failed     = progress.get('failed', {})
    # Only keep slice_meta for patients confirmed on disk
    slice_meta = [s for s in progress.get('slice_meta', [])
                  if s['patient_id'] in completed]
    print(f'RESUMING — {len(completed)} confirmed on disk, {len(failed)} failed')
else:
    completed  = set()
    failed     = {}
    slice_meta = []
    print('Starting fresh')

# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
def get_nii_path(patient_dir, patient_id, seq_name):
    """
    Handles nested folder structure:
    UCSF-PDGM-0004/
      UCSF-PDGM-0004_T1.nii/
        UCSF-PDGM-0004_T1.nii  <-- actual file
    Also handles flat structure as fallback.
    """
    filename = f'{patient_id}_{seq_name}.nii'
    nested   = os.path.join(patient_dir, filename, filename)
    flat     = os.path.join(patient_dir, filename)
    if os.path.isfile(nested):
        return nested
    elif os.path.isfile(flat):
        return flat
    return None

def load_volume(path):
    img = nib.load(path)
    return img.get_fdata(dtype=np.float32)

def clip_and_normalize(vol, brain_mask):
    brain_voxels = vol[brain_mask > 0]
    if brain_voxels.size == 0:
        return vol
    # Clip to 1st-99th percentile within brain mask
    p1  = np.percentile(brain_voxels, 1)
    p99 = np.percentile(brain_voxels, 99)
    vol = np.clip(vol, p1, p99)
    # Z-score normalize within brain mask
    brain_voxels = vol[brain_mask > 0]
    mean = brain_voxels.mean()
    std  = brain_voxels.std()
    if std < 1e-8:
        vol = np.zeros_like(vol)
    else:
        vol = (vol - mean) / std
    vol[brain_mask == 0] = 0.0
    return vol

def select_tumor_slices(tumor_mask):
    n_slices   = tumor_mask.shape[2]
    slice_area = tumor_mask.shape[0] * tumor_mask.shape[1]
    selected   = []
    for s in range(n_slices):
        coverage = (tumor_mask[:, :, s] > 0).sum() / slice_area
        if coverage >= TUMOR_THR:
            selected.append(s)
    return selected

def resize_slice(slc):
    return resize(slc, (SLICE_SIZE, SLICE_SIZE),
                  order=1, mode='constant',
                  anti_aliasing=True,
                  preserve_range=True).astype(np.float32)

def preprocess_patient(patient_id, patient_dir):
    try:
        # Resolve file paths
        tumor_path = get_nii_path(patient_dir, patient_id, TUMOR_SEQ)
        brain_path = get_nii_path(patient_dir, patient_id, BRAIN_SEQ)

        if tumor_path is None:
            return None, 0, f'Cannot find tumor_segmentation.nii'
        if brain_path is None:
            return None, 0, f'Cannot find brain_segmentation.nii'

        # Load and binarize masks
        tumor_mask = load_volume(tumor_path)
        brain_mask = load_volume(brain_path)
        tumor_mask = (tumor_mask > 0).astype(np.float32)
        brain_mask = (brain_mask > 0).astype(np.float32)

        # Select tumor-containing axial slices
        selected = select_tumor_slices(tumor_mask)
        if len(selected) == 0:
            return None, 0, 'No tumor slices above 0.1% coverage threshold'

        # Load, normalize, extract slices for each sequence
        channel_arrays = []
        for seq in TRACK_A_SEQS:
            seq_path = get_nii_path(patient_dir, patient_id, seq)
            if seq_path is None:
                return None, 0, f'Cannot find {seq}.nii'
            vol = load_volume(seq_path)
            vol = clip_and_normalize(vol, brain_mask)
            seq_slices = np.stack(
                [resize_slice(vol[:, :, s]) for s in selected],
                axis=0
            )  # shape: (n_slices, 224, 224)
            channel_arrays.append(seq_slices)

        # Stack into Track A tensor: (n_slices, 7, 224, 224)
        # Track B = trackA[:, :4, :, :] — derived at runtime, not saved separately
        trackA = np.stack(channel_arrays, axis=1).astype(np.float16)

        return trackA, len(selected), None

    except Exception as e:
        return None, 0, str(e)

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
remaining = [p for p in df_labels['patient_id'].tolist() if p not in completed]

print(f'Patients remaining : {len(remaining)}')
print(f'Saving to          : {WORKING}/trackA_slices/')
print(f'Track B derived at runtime from first 4 channels of Track A')
print('='*60)

for i, patient_id in enumerate(remaining):
    row         = df_labels[df_labels['patient_id'] == patient_id].iloc[0]
    patient_dir = row['folder_path']
    fold        = int(row['fold'])
    idh_label   = int(row['IDH_binary'])

    trackA, n_slices, error = preprocess_patient(patient_id, patient_dir)

    if error:
        failed[patient_id] = error
        print(f'  [{i+1}/{len(remaining)}] FAIL {patient_id} — {error}')
    else:
        np.save(f'{WORKING}/trackA_slices/{patient_id}.npy', trackA)
        for s in range(n_slices):
            slice_meta.append({
                'patient_id': patient_id,
                'slice_idx':  s,
                'fold':       fold,
                'idh_label':  idh_label
            })
        completed.add(patient_id)
        # Remove from failed if it was previously failed
        failed.pop(patient_id, None)
        print(f'  [{i+1}/{len(remaining)}] OK  {patient_id} — {n_slices} slices | fold {fold} | IDH {idh_label}')

    # Save progress after every patient
    with open(progress_path, 'w') as f:
        json.dump({
            'completed':  list(completed),
            'failed':     failed,
            'slice_meta': slice_meta
        }, f)

print()
print('='*60)
print(f'Completed : {len(completed)}/{len(df_labels)}')
print(f'Failed    : {len(failed)}')
if failed:
    print('Failed patients:')
    for pid, reason in failed.items():
        print(f'  {pid}: {reason}')
if len(completed) == len(df_labels):
    print('All patients done — proceed to Cell 5.')
else:
    print('Re-run Cell 1 then Cell 4 to continue.')



# ── SUMMARY ───────────────────────────────────────────────────────────────────
df_slices = pd.DataFrame(slice_meta)
df_slices.to_csv(f'{WORKING}/slice_metadata.csv', index=False)

print()
print('='*60)
print('PREPROCESSING SUMMARY')
print('='*60)
print(f'Patients processed     : {len(completed)}')
print(f'Patients failed        : {len(failed)}')
print(f'Total slices           : {len(df_slices)}')
print(f'Average slices/patient : {len(df_slices)/len(completed):.1f}')
print()
print('Slices per fold:')
for fold in range(5):
    fold_df = df_slices[df_slices['fold'] == fold]
    wt = (fold_df['idh_label'] == 0).sum()
    mt = (fold_df['idh_label'] == 1).sum()
    print(f'  Fold {fold}: {len(fold_df):>5} slices — {wt} wildtype / {mt} mutant')
print()
print('Files in /kaggle/working/:')
print('  labels.csv          — patient ID, IDH label, grade, fold')
print('  patient_splits.json — fold assignments per patient')
print('  slice_metadata.csv  — slice to patient mapping')
print('  trackA_slices/      — 495 .npy files, float16, (n,7,224,224)')
print('  Track B = trackA[:, :4, :, :] derived at runtime')

SystemExit: OLD CELL 4 RETIRED — use new Cell 4 (verify mount) instead

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [10]:
raise SystemExit("Tensors already saved to kaggle dataset with this cell")
import subprocess, json, os, shutil

WORKING = '/kaggle/working'
SLICES_DIR = f'{WORKING}/trackA_slices'
TENSORS_DS = 'adesaladaniel/glioma-idh-tensors'

# 1. Clean up everything EXCEPT the slices to free every possible megabyte
for folder in ['cnn_trackA', 'cnn_trackB', 'kaggle_sync', 'upload_batch']:
    path = os.path.join(WORKING, folder)
    if os.path.exists(path):
        shutil.rmtree(path)

# 2. Check if metadata exists, if not, create it
metadata_path = os.path.join(SLICES_DIR, 'dataset-metadata.json')
with open(metadata_path, 'w') as f:
    json.dump({
        'title': 'Glioma IDH Tensors',
        'id': TENSORS_DS,
        'licenses': [{'name': 'other'}],
        'isPrivate': True
    }, f)

print("🚀 Starting the BIG upload using CLI TAR mode...")
print("This will take a while. Do not close this tab.")

# 3. Use TAR mode - it is much more stable for 20GB+ datasets
result = subprocess.run(
    ['kaggle', 'datasets', 'version', 
     '-p', SLICES_DIR, 
     '-m', 'Full 0.1 percent threshold tensors', 
     '--dir-mode', 'tar'], 
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ SUCCESS! Tensors are now permanent on Kaggle.")
    print(result.stdout)
else:
    print("❌ CLI Upload failed.")
    print("Error details:", result.stderr)

SystemExit: Tensors already saved to kaggle dataset with this cell

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
# ── TENSOR UPLOAD — COMPLETED ─────────────────────────────────────────────────
# All 495 preprocessed tensor files were uploaded to Kaggle on 2026-06-26.
#
# Permanent storage dataset:
#   adesaladaniel/glioma-idh-tensors
#
# Contents:
#   495 .npy files — one per patient
#   Shape per file: (n_slices, 7, 224, 224) float16
#   Threshold: 0.1% tumor coverage
#   Total slices: 28,881
#   Total size: ~20 GB
#
# To access in future sessions:
#   Use new Cell 4 (downloader version) — downloads in 2-3 minutes
#   No reprocessing ever needed again
#
# Upload method: kaggle datasets version --dir-mode tar
# Upload date: 2026-06-26
# Uploaded by: Daniel Adesala
print("Tensors permanently stored at: adesaladaniel/glioma-idh-tensors ✅")

Tensors permanently stored at: adesaladaniel/glioma-idh-tensors ✅


In [6]:
# ── CELL 4: VERIFY TENSOR DATASET IS MOUNTED ─────────────────────────────────
# Tensors are permanently stored at:
#   adesaladaniel/glioma-idh-tensors
#
# Mounted as input dataset at:
#   /kaggle/input/glioma-idh-tensors/
#
# This uses ZERO working directory disk space.
# No preprocessing needed. No download needed.
# Just verify the mount and proceed to Cell 5.

import os
import numpy as np
import pandas as pd
import shutil

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

print('='*60)
print('CELL 4 — TENSOR VERIFICATION')
print('='*60)

# ── CHECK TENSORS ARE ACCESSIBLE ──────────────────────────────────────────────
if not os.path.exists(SLICES_DIR):
    print('❌ ERROR: Tensor dataset not mounted.')
    print('   Go to notebook settings → Add Data')
    print('   Search for: adesaladaniel/glioma-idh-tensors')
    print('   Add it as input dataset and restart session.')
    raise SystemExit

npy_files = sorted([
    f for f in os.listdir(SLICES_DIR)
    if f.endswith('.npy')
])

print(f'Tensor location : {SLICES_DIR}')
print(f'Tensors found   : {len(npy_files)}/495')

# ── SPOT CHECK FIRST FILE ─────────────────────────────────────────────────────
if npy_files:
    sample_path = os.path.join(SLICES_DIR, npy_files[0])
    sample = np.load(sample_path, mmap_mode='r')
    print(f'Sample file     : {npy_files[0]}')
    print(f'Sample shape    : {sample.shape}')
    print(f'Sample dtype    : {sample.dtype}')
    print(f'Channels        : {sample.shape[1]} (expect 7 for Track A)')

# ── SPOT CHECK LAST FILE ──────────────────────────────────────────────────────
if len(npy_files) > 1:
    last_path = os.path.join(SLICES_DIR, npy_files[-1])
    last = np.load(last_path, mmap_mode='r')
    print(f'Last file       : {npy_files[-1]}')
    print(f'Last shape      : {last.shape}')

# ── DISK USAGE ────────────────────────────────────────────────────────────────
print()
total, used, free = shutil.disk_usage(WORKING)
print(f'Working directory disk usage:')
print(f'  Total : {total/1e9:.1f} GB')
print(f'  Used  : {used/1e9:.2f} GB')
print(f'  Free  : {free/1e9:.2f} GB')

# ── FINAL VERDICT ─────────────────────────────────────────────────────────────
print()
if len(npy_files) == 495:
    print('✅ All 495 tensors accessible')
    print('✅ Zero working directory space used for tensors')
    print('✅ Ready for Cell 5')
else:
    print(f'⚠️  Only {len(npy_files)}/495 tensors found')
    print('Check that glioma-idh-tensors is properly mounted')

CELL 4 — TENSOR VERIFICATION
Tensor location : /kaggle/input/datasets/adesaladaniel/glioma-idh-tensors
Tensors found   : 495/495
Sample file     : UCSF-PDGM-0004.npy
Sample shape    : (42, 7, 224, 224)
Sample dtype    : float16
Channels        : 7 (expect 7 for Track A)
Last file       : UCSF-PDGM-0541.npy
Last shape      : (82, 7, 224, 224)

Working directory disk usage:
  Total : 21.0 GB
  Used  : 0.00 GB
  Free  : 20.94 GB

✅ All 495 tensors accessible
✅ Zero working directory space used for tensors
✅ Ready for Cell 5


In [7]:
# ── CELL 4 AUDIT: FULL TENSOR INTEGRITY CHECK ────────────────────────────────
# Verifies:
#   - All 495 patients present
#   - Slice counts match expected (~28,881 total)
#   - Fold distribution matches patient_splits.json
#   - IDH label alignment correct
#   - No corrupted files

import os
import json
import numpy as np
import pandas as pd
import shutil

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

print('='*60)
print('CELL 4 AUDIT — TENSOR INTEGRITY CHECK')
print('='*60)

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))
fold_map  = dict(zip(df_labels['patient_id'],
                     df_labels['fold'].astype(int)))

# ── COUNT FILES ───────────────────────────────────────────────────────────────
npy_files = sorted([
    f for f in os.listdir(SLICES_DIR)
    if f.endswith('.npy')
])
patient_ids_on_disk = set(f.replace('.npy', '') for f in npy_files)
patient_ids_in_labels = set(df_labels['patient_id'].tolist())

missing_from_disk   = patient_ids_in_labels - patient_ids_on_disk
extra_on_disk       = patient_ids_on_disk - patient_ids_in_labels

print(f'\nFile count check:')
print(f'  Files on disk    : {len(npy_files)}')
print(f'  Patients in CSV  : {len(df_labels)}')
print(f'  Missing from disk: {len(missing_from_disk)}')
print(f'  Extra on disk    : {len(extra_on_disk)}')

if missing_from_disk:
    print(f'  Missing patients : {sorted(missing_from_disk)[:5]}')

# ── SLICE COUNT AND FOLD DISTRIBUTION ─────────────────────────────────────────
total_slices = 0
fold_counts  = {fold: {'wt': 0, 'mt': 0, 'total': 0} for fold in range(5)}
shape_issues = []
dtype_issues = []

print(f'\nChecking all {len(npy_files)} tensor files...')

for npy_file in npy_files:
    pid  = npy_file.replace('.npy', '')
    path = os.path.join(SLICES_DIR, npy_file)

    try:
        arr = np.load(path, mmap_mode='r')
    except Exception as e:
        print(f'  ❌ Corrupt file: {npy_file} — {e}')
        continue

    n_slices = arr.shape[0]
    total_slices += n_slices

    # Check shape
    if len(arr.shape) != 4 or arr.shape[1] != 7 or arr.shape[2] != 224:
        shape_issues.append((npy_file, arr.shape))

    # Check dtype
    if arr.dtype != np.float16:
        dtype_issues.append((npy_file, arr.dtype))

    # Fold distribution
    if pid in fold_map:
        fold = fold_map[pid]
        fold_counts[fold]['total'] += n_slices
        if label_map.get(pid, 0) == 0:
            fold_counts[fold]['wt'] += n_slices
        else:
            fold_counts[fold]['mt'] += n_slices

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print()
print('='*60)
print('AUDIT RESULTS')
print('='*60)
print(f'Patients    : {len(npy_files)}/495')
print(f'Total slices: {total_slices}')
print(f'Avg slices  : {total_slices/max(len(npy_files),1):.1f} per patient')
print()

print('Slices per fold:')
for fold in range(5):
    fc = fold_counts[fold]
    print(f'  Fold {fold}: {fc["total"]:>5} slices — '
          f'{fc["wt"]} wildtype / {fc["mt"]} mutant')

print()
print('Shape issues  :', len(shape_issues))
if shape_issues:
    for f, s in shape_issues[:3]:
        print(f'  {f}: {s}')

print('Dtype issues  :', len(dtype_issues))
if dtype_issues:
    for f, d in dtype_issues[:3]:
        print(f'  {f}: {d}')

print()
total_disk, used_disk, free_disk = shutil.disk_usage(WORKING)
print(f'Disk: {used_disk/1e9:.2f} GB used / '
      f'{total_disk/1e9:.1f} GB total / '
      f'{free_disk/1e9:.2f} GB free')

# ── FINAL VERDICT ─────────────────────────────────────────────────────────────
print()
print('='*60)
all_good = (
    len(npy_files) == 495 and
    len(shape_issues) == 0 and
    len(dtype_issues) == 0 and
    len(missing_from_disk) == 0
)

if all_good:
    print('✅ ALL CHECKS PASSED')
    print(f'✅ {len(npy_files)} patients')
    print(f'✅ {total_slices} slices')
    print(f'✅ All shapes correct (n, 7, 224, 224)')
    print(f'✅ All dtypes correct (float16)')
    print(f'✅ {free_disk/1e9:.2f} GB free for model training')
    print()
    print('Ready for Cell 5 🚀')
else:
    print('⚠️  SOME CHECKS FAILED — review issues above')

CELL 4 AUDIT — TENSOR INTEGRITY CHECK

File count check:
  Files on disk    : 495
  Patients in CSV  : 495
  Missing from disk: 0
  Extra on disk    : 0

Checking all 495 tensor files...

AUDIT RESULTS
Patients    : 495/495
Total slices: 28881
Avg slices  : 58.3 per patient

Slices per fold:
  Fold 0:  5838 slices — 4738 wildtype / 1100 mutant
  Fold 1:  5911 slices — 4832 wildtype / 1079 mutant
  Fold 2:  5627 slices — 4579 wildtype / 1048 mutant
  Fold 3:  5823 slices — 4761 wildtype / 1062 mutant
  Fold 4:  5682 slices — 4508 wildtype / 1174 mutant

Shape issues  : 0
Dtype issues  : 0

Disk: 0.00 GB used / 21.0 GB total / 20.94 GB free

✅ ALL CHECKS PASSED
✅ 495 patients
✅ 28881 slices
✅ All shapes correct (n, 7, 224, 224)
✅ All dtypes correct (float16)
✅ 20.94 GB free for model training

Ready for Cell 5 🚀


## Cell 5: CNN Training

**Track A (ResNet18 7-channel)**

In [10]:
# ── CELL 5: CNN TRAINING — TRACK A (ResNet18 7-channel) ──────────────────────
# Track A = 7 channels: T1, T1c, T2, FLAIR, ADC, FA, MD
# FIXED: progress JSON now restored from Kaggle before resume check

import os, json, shutil, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from kaggle_secrets import UserSecretsClient
from kaggle.api.kaggle_api_extended import KaggleApi

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

os.makedirs(f'{WORKING}/cnn_trackA', exist_ok=True)
os.makedirs(f'{WORKING}/cnn_trackB', exist_ok=True)

# ── KAGGLE API SETUP ──────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
kaggle_api = KaggleApi()
kaggle_api.authenticate()

MODELS_DATASET = 'adesaladaniel/glioma-idh-models'
print(f'Kaggle API ready — saving to {MODELS_DATASET}')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 7
N_CLASSES    = 2
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
PATIENCE     = 10
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT = 3.81
SEED         = 42
N_FOLDS      = 5
SAVE_DIR     = f'{WORKING}/cnn_trackA'
TRACKB_DIR   = f'{WORKING}/cnn_trackB'

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device     : {DEVICE}')
print(f'Track      : A — {N_CHANNELS} channels')
print(f'Class wt   : {CLASS_WEIGHT}')
print(f'Slices dir : {SLICES_DIR}')

# ── SYNC EXISTING KAGGLE FILES ────────────────────────────────────────────────
def sync_existing_files():
    """
    Downloads ALL files from glioma-idh-models to working root.
    This includes both Track A and Track B .pth and .json files.
    FIXED: also routes .json progress files to correct locations.
    """
    print('\nSyncing existing model files from Kaggle...')
    sync_dir = f'{WORKING}/kaggle_sync'
    os.makedirs(sync_dir, exist_ok=True)

    result = subprocess.run(
        ['kaggle', 'datasets', 'download',
         MODELS_DATASET, '--path', sync_dir, '--unzip'],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        synced = [f for f in os.listdir(sync_dir)
                  if not f.startswith('.') and
                  f != 'dataset-metadata.json']
        print(f'Downloaded {len(synced)} existing files:')
        for f in sorted(synced):
            src = os.path.join(sync_dir, f)
            dst = os.path.join(WORKING, f)
            shutil.copy(src, dst)
            print(f'  ✅ {f}')
            # Route .pth weights to subdirectories
            if 'trackB' in f and f.endswith('.pth'):
                shutil.copy(src, os.path.join(
                    TRACKB_DIR, f.replace('cnn_trackB_', '')))
            if 'trackA' in f and f.endswith('.pth'):
                shutil.copy(src, os.path.join(
                    SAVE_DIR, f.replace('cnn_trackA_', '')))
    else:
        print('No existing files found — starting fresh')

    shutil.rmtree(sync_dir, ignore_errors=True)
    print('Sync complete.\n')

sync_existing_files()

# ── SAFE COMMIT ───────────────────────────────────────────────────────────────
def safe_commit_to_kaggle(message):
    commit_dir = f'{WORKING}/kaggle_commit'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir, 'dataset-metadata.json'), 'w') as f:
        json.dump({
            'title':    'Glioma IDH Models',
            'id':       MODELS_DATASET,
            'licenses': [{'name': 'other'}],
            'isPrivate': True
        }, f)

    # Symlink all cnn_track* files — zero disk duplication
    for f in os.listdir(WORKING):
        if f.startswith('cnn_track') and (
                f.endswith('.pth') or f.endswith('.json')):
            src = os.path.join(WORKING, f)
            lnk = os.path.join(commit_dir, f)
            if os.path.exists(src) and not os.path.lexists(lnk):
                os.symlink(os.path.abspath(src), lnk)

    result = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir,
         '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f'  ✅ Saved to Kaggle: {message}')
    else:
        print(f'  ⚠️  Save failed: {result.stderr[:300]}')

    shutil.rmtree(commit_dir, ignore_errors=True)
    return result.returncode == 0

# ── DATASET ───────────────────────────────────────────────────────────────────
class GliomaDataset(Dataset):
    def __init__(self, patient_ids, df_labels, slices_dir,
                 n_channels=7, augment=False):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.augment    = augment
        self.items      = []
        label_map = dict(zip(df_labels['patient_id'],
                             df_labels['IDH_binary'].astype(int)))
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s, label_map[pid]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s, label = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            if torch.rand(1) > 0.5:
                angle = (torch.rand(1).item() - 0.5) * 30
                x = TF.rotate(x, angle)
            x = x * (0.9 + torch.rand(1).item() * 0.2)
            x = x + torch.randn_like(x) * 0.02
        return x, label, pid

# ── MODEL ─────────────────────────────────────────────────────────────────────
def build_model(n_channels, n_classes):
    model = resnet18(weights='IMAGENET1K_V1')
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    with torch.no_grad():
        avg_w = model.conv1.weight.mean(dim=1, keepdim=True)
        new_conv.weight = nn.Parameter(avg_w.repeat(1, n_channels, 1, 1))
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256), nn.ReLU(),
        nn.Dropout(0.5), nn.Linear(256, n_classes)
    )
    return model

# ── TRAIN ONE EPOCH ───────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

# ── EVALUATE ──────────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device, label_map):
    model.eval()
    total_loss, patient_probs = 0, {}
    all_slice_probs, all_slice_labels = [], []

    with torch.no_grad():
        for x, y, pids in loader:
            x, y = x.to(device), y.to(device)
            out  = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = y.cpu().numpy()
            all_slice_probs.extend(probs)
            all_slice_labels.extend(labels)
            for prob, pid in zip(probs, pids):
                patient_probs.setdefault(pid, []).append(float(prob))

    pid_list       = list(patient_probs.keys())
    pid_mean_probs = [np.mean(patient_probs[p]) for p in pid_list]
    pid_labels     = [label_map[p] for p in pid_list]
    pid_preds      = [1 if p >= 0.5 else 0 for p in pid_mean_probs]

    patient_auc = roc_auc_score(pid_labels, pid_mean_probs)
    slice_auc   = roc_auc_score(all_slice_labels, all_slice_probs)

    cm = confusion_matrix(pid_labels, pid_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    metrics = {
        'patient_auc': round(float(patient_auc), 4),
        'slice_auc':   round(float(slice_auc), 4),
        'accuracy':    round(float(accuracy_score(pid_labels, pid_preds)), 4),
        'sensitivity': round(float(recall_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'specificity': round(float(tn / (tn + fp)) if (tn+fp) > 0 else 0.0, 4),
        'f1':          round(float(f1_score(
            pid_labels, pid_preds, zero_division=0)), 4),
        'brier':       round(float(brier_score_loss(
            pid_labels, pid_mean_probs)), 4),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
    }
    return total_loss / len(loader.dataset), metrics, patient_probs

# ── RELOAD PROGRESS ───────────────────────────────────────────────────────────
cnn_progress_path = f'{WORKING}/cnn_trackA_progress.json'

if os.path.exists(cnn_progress_path):
    with open(cnn_progress_path) as f:
        cnn_progress = json.load(f)
    completed_folds = cnn_progress.get('completed_folds', [])
    fold_results    = cnn_progress.get('fold_results', {})
    all_val_preds   = cnn_progress.get('all_val_preds', {})
    print(f'RESUMING CNN Track A — completed folds: {completed_folds}')
else:
    completed_folds = []
    fold_results    = {}
    all_val_preds   = {}
    print('Starting CNN Track A fresh — Fold 0 first')

# Restore weights for any completed folds
for fold in completed_folds:
    local_path   = f'{SAVE_DIR}/fold{fold}_best.pth'
    working_copy = f'{WORKING}/cnn_trackA_fold{fold}_best.pth'
    if not os.path.exists(local_path) and os.path.exists(working_copy):
        shutil.copy(working_copy, local_path)
        print(f'Restored Track A fold {fold} weights from working root')

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map     = dict(zip(df_labels['patient_id'],
                         df_labels['IDH_binary'].astype(int)))
class_weights = torch.tensor([1.0, CLASS_WEIGHT],
                              dtype=torch.float32).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── 5-FOLD TRAINING LOOP ──────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — 5-FOLD CROSS VALIDATION')
print(f'{"="*60}')

for fold in range(N_FOLDS):
    if fold in completed_folds:
        r = fold_results[str(fold)]
        print(f'\nFold {fold} — already done '
              f'(AUC {r["best_auc"]:.4f}, epoch {r["best_epoch"]}), skipping')
        continue

    print(f'\nFOLD {fold}/{N_FOLDS-1}')
    print('-' * 40)

    val_patients   = splits[f'fold_{fold}']
    train_patients = [p for f2 in range(N_FOLDS) if f2 != fold
                      for p in splits[f'fold_{f2}']]

    train_ds = GliomaDataset(train_patients, df_labels, SLICES_DIR,
                             n_channels=N_CHANNELS, augment=True)
    val_ds   = GliomaDataset(val_patients,   df_labels, SLICES_DIR,
                             n_channels=N_CHANNELS, augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f'Train : {len(train_patients)} patients / {len(train_ds)} slices')
    print(f'Val   : {len(val_patients)} patients / {len(val_ds)} slices')

    model = build_model(N_CHANNELS, N_CLASSES).to(DEVICE)

    # Freeze backbone for first 5 epochs
    for name, param in model.named_parameters():
        if not name.startswith('fc'):
            param.requires_grad = False

    optimizer = optim.AdamW([
        {'params': model.fc.parameters(),  'lr': LR_HEAD},
        {'params': [p for n, p in model.named_parameters()
                    if not n.startswith('fc')], 'lr': LR_BACKBONE}
    ], weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_auc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    best_metrics = {}
    history      = []

    for epoch in range(1, MAX_EPOCHS + 1):

        # Unfreeze backbone at epoch 6
        if epoch == 6:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.fc.parameters(), 'lr': LR_HEAD},
                {'params': [p for n, p in model.named_parameters()
                            if not n.startswith('fc')], 'lr': LR_BACKBONE}
            ], weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - 5)
            print(f'  Epoch {epoch}: backbone unfrozen, optimizer reset')

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE)
        val_loss, metrics, _ = evaluate(
            model, val_loader, criterion, DEVICE, label_map)
        scheduler.step()

        history.append({
            'epoch': epoch,
            'train_loss': round(float(train_loss), 4),
            'val_loss':   round(float(val_loss), 4),
            **metrics
        })

        print(
            f'  Epoch {epoch:02d} | '
            f'loss: {train_loss:.4f}/{val_loss:.4f} | '
            f'pt_AUC: {metrics["patient_auc"]:.4f} | '
            f'sl_AUC: {metrics["slice_auc"]:.4f} | '
            f'sens: {metrics["sensitivity"]:.4f} | '
            f'spec: {metrics["specificity"]:.4f} | '
            f'F1: {metrics["f1"]:.4f} | '
            f'Brier: {metrics["brier"]:.4f}'
        )

        if metrics['patient_auc'] > best_val_auc:
            best_val_auc = metrics['patient_auc']
            best_epoch   = epoch
            best_metrics = metrics.copy()
            patience_ctr = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/fold{fold}_best.pth')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Early stopping — best epoch {best_epoch}, '
                      f'AUC {best_val_auc:.4f}')
                break

    # Reload best weights and get final patient-level predictions
    print(f'  Reloading best weights (epoch {best_epoch})...')
    model.load_state_dict(torch.load(
        f'{SAVE_DIR}/fold{fold}_best.pth', map_location=DEVICE))
    _, best_metrics, patient_probs = evaluate(
        model, val_loader, criterion, DEVICE, label_map)

    for pid, probs in patient_probs.items():
        all_val_preds[pid] = {
            'mean_prob': float(np.mean(probs)),
            'label':     int(label_map[pid]),
            'fold':      fold
        }

    fold_results[str(fold)] = {
        'best_epoch':   best_epoch,
        'best_auc':     round(best_val_auc, 4),
        'best_metrics': best_metrics,
        'history':      history
    }
    completed_folds.append(fold)

    print(f'\n  ── Fold {fold} Summary (epoch {best_epoch}) ──')
    print(f'    Patient AUC  : {best_metrics["patient_auc"]:.4f}')
    print(f'    Sensitivity  : {best_metrics["sensitivity"]:.4f}')
    print(f'    Specificity  : {best_metrics["specificity"]:.4f}')
    print(f'    F1           : {best_metrics["f1"]:.4f}')
    print(f'    Accuracy     : {best_metrics["accuracy"]:.4f}')
    print(f'    Brier        : {best_metrics["brier"]:.4f}')
    print(f'    TP:{best_metrics["tp"]} TN:{best_metrics["tn"]} '
          f'FP:{best_metrics["fp"]} FN:{best_metrics["fn"]}')

    # ── SAVE PROGRESS ─────────────────────────────────────────────────────────
    with open(cnn_progress_path, 'w') as f:
        json.dump({
            'completed_folds': completed_folds,
            'fold_results':    fold_results,
            'all_val_preds':   all_val_preds
        }, f)

    # Copy to working root so commit picks it up
    shutil.copy(f'{SAVE_DIR}/fold{fold}_best.pth',
                f'{WORKING}/cnn_trackA_fold{fold}_best.pth')

    # Commit everything (Track A + Track B) to Kaggle
    safe_commit_to_kaggle(
        f'CNN Track A fold {fold} complete — AUC {best_val_auc:.4f}')

    del model, optimizer, scheduler, train_ds, val_ds
    del train_loader, val_loader
    torch.cuda.empty_cache()

    _, used, free = shutil.disk_usage(WORKING)
    print(f'  Disk: {used/1e9:.2f} GB used / {free/1e9:.2f} GB free')

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK A — COMPLETE')
print(f'{"="*60}')

aucs  = [fold_results[str(f)]['best_auc']              for f in range(N_FOLDS)]
sens  = [fold_results[str(f)]['best_metrics']['sensitivity'] for f in range(N_FOLDS)]
spec  = [fold_results[str(f)]['best_metrics']['specificity'] for f in range(N_FOLDS)]
f1s   = [fold_results[str(f)]['best_metrics']['f1']         for f in range(N_FOLDS)]
brier = [fold_results[str(f)]['best_metrics']['brier']       for f in range(N_FOLDS)]
acc   = [fold_results[str(f)]['best_metrics']['accuracy']    for f in range(N_FOLDS)]

print(f'\nPer-fold AUC     : {[round(a,4) for a in aucs]}')
print(f'Mean AUC         : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'Mean Sensitivity : {np.mean(sens):.4f} ± {np.std(sens):.4f}')
print(f'Mean Specificity : {np.mean(spec):.4f} ± {np.std(spec):.4f}')
print(f'Mean F1          : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'Mean Accuracy    : {np.mean(acc):.4f} ± {np.std(acc):.4f}')
print(f'Mean Brier       : {np.mean(brier):.4f} ± {np.std(brier):.4f}')
print(f'\nPatients with OOF predictions : {len(all_val_preds)} (expect 495)')
print(f'Weights + progress saved to   : {MODELS_DATASET}')
print(f'\n✅ Ready for Cell 6 — CNN Track B (Fold 4 only needed)')

Kaggle API ready — saving to adesaladaniel/glioma-idh-models
Device     : cuda
Track      : A — 7 channels
Class wt   : 3.81
Slices dir : /kaggle/input/datasets/adesaladaniel/glioma-idh-tensors

Syncing existing model files from Kaggle...
Downloaded 11 existing files:
  ✅ cnn_trackA_fold0_best.pth
  ✅ cnn_trackA_fold1_best.pth
  ✅ cnn_trackA_fold2_best.pth
  ✅ cnn_trackA_fold3_best.pth
  ✅ cnn_trackA_fold4_best.pth
  ✅ cnn_trackA_progress.json
  ✅ cnn_trackB_fold0_best.pth
  ✅ cnn_trackB_fold1_best.pth
  ✅ cnn_trackB_fold2_best.pth
  ✅ cnn_trackB_fold3_best.pth
  ✅ cnn_trackB_progress.json
Sync complete.

RESUMING CNN Track A — completed folds: [0, 1, 2, 3, 4]

CNN TRACK A — 5-FOLD CROSS VALIDATION

Fold 0 — already done (AUC 0.9709, epoch 18), skipping

Fold 1 — already done (AUC 0.9411, epoch 8), skipping

Fold 2 — already done (AUC 0.9554, epoch 6), skipping

Fold 3 — already done (AUC 0.9982, epoch 11), skipping

Fold 4 — already done (AUC 0.9615, epoch 14), skipping

CNN TRACK A —

## Cell 6: CNN Training

**Track B (ResNet18 4-channel)**

In [11]:
# ── CELL 6: CNN TRAINING — TRACK B (ResNet18 4-channel) ──────────────────────
# Track B = first 4 channels only: T1, T1c, T2, FLAIR
# Derived at runtime: arr[s, :4, :, :]
#
# INFRASTRUCTURE:
# - Tensors read from /kaggle/input/datasets/adesaladaniel/glioma-idh-tensors
# - Models saved to adesaladaniel/glioma-idh-models
# - Safe commit: Track A + Track B saved together (no overwrite)
#
# Identical training protocol to Cell 5 (Track A):
# - Backbone frozen epochs 1-5, unfrozen from epoch 6
# - Differential LR: head 1e-3, backbone 1e-4
# - AdamW + CosineAnnealingLR
# - Early stopping patience=10 on patient-level AUC
# - Best-epoch re-evaluation before saving predictions
# - Symlink-based cumulative Kaggle commit after every fold

import os, json, shutil, subprocess, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from kaggle_secrets import UserSecretsClient
from kaggle.api.kaggle_api_extended import KaggleApi

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'

os.makedirs(f'{WORKING}/cnn_trackA', exist_ok=True)
os.makedirs(f'{WORKING}/cnn_trackB', exist_ok=True)

# ── KAGGLE API SETUP ──────────────────────────────────────────────────────────
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
kaggle_api = KaggleApi()
kaggle_api.authenticate()

MODELS_DATASET = 'adesaladaniel/glioma-idh-models'
print(f'Kaggle API ready — saving to {MODELS_DATASET}')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 4
N_CLASSES    = 2
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
PATIENCE     = 10
LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-4
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT = 3.81
SEED         = 42
N_FOLDS      = 5
SAVE_DIR     = f'{WORKING}/cnn_trackB'
TRACKA_DIR   = f'{WORKING}/cnn_trackA'

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Device     : {DEVICE}')
print(f'Track      : B — {N_CHANNELS} channels (T1, T1c, T2, FLAIR)')
print(f'Class wt   : {CLASS_WEIGHT}')
print(f'Slices dir : {SLICES_DIR}')

# ── SYNC EXISTING KAGGLE FILES ────────────────────────────────────────────────
def sync_existing_files():
    print('\nSyncing existing model files from Kaggle...')
    sync_dir = f'{WORKING}/kaggle_sync'
    os.makedirs(sync_dir, exist_ok=True)

    result = subprocess.run(
        ['kaggle', 'datasets', 'download',
         MODELS_DATASET, '--path', sync_dir, '--unzip'],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        synced = [f for f in os.listdir(sync_dir)
                  if not f.startswith('.') and
                  f != 'dataset-metadata.json']
        print(f'Downloaded {len(synced)} existing files:')
        for f in sorted(synced):
            src = os.path.join(sync_dir, f)
            dst = os.path.join(WORKING, f)
            shutil.copy(src, dst)
            print(f'  ✅ {f}')
            if 'trackA' in f and f.endswith('.pth'):
                fold_name = f.replace('cnn_trackA_', '')
                shutil.copy(src, os.path.join(TRACKA_DIR, fold_name))
            if 'trackB' in f and f.endswith('.pth'):
                fold_name = f.replace('cnn_trackB_', '')
                shutil.copy(src, os.path.join(SAVE_DIR, fold_name))
    else:
        print('No existing files found — starting fresh')

    shutil.rmtree(sync_dir, ignore_errors=True)
    print('Sync complete.\n')

sync_existing_files()

# ── SAFE COMMIT FUNCTION ──────────────────────────────────────────────────────
def safe_commit_to_kaggle(message):
    """
    Commits all model files using symlinks — zero disk duplication.
    Includes both Track A and Track B files in every commit.
    """
    commit_dir = f'{WORKING}/kaggle_commit'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir, 'dataset-metadata.json'), 'w') as f:
        json.dump({
            'title':     'Glioma IDH Models',
            'id':        MODELS_DATASET,
            'licenses':  [{'name': 'other'}],
            'isPrivate': True
        }, f)

    # Symlink all model files from working directory
    for f in os.listdir(WORKING):
        if f.startswith('cnn_track') and (f.endswith('.pth') or
                                           f.endswith('.json')):
            src = os.path.join(WORKING, f)
            lnk = os.path.join(commit_dir, f)
            if os.path.exists(src) and not os.path.lexists(lnk):
                os.symlink(os.path.abspath(src), lnk)

    result = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir,
         '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f'  Saved to Kaggle: {message} ✓')
    else:
        print(f'  WARNING: Save failed: {result.stderr[:300]}')

    shutil.rmtree(commit_dir, ignore_errors=True)
    return result.returncode == 0

# ── DATASET (4 CHANNELS ONLY) ─────────────────────────────────────────────────
class GliomaDataset(Dataset):
    """
    Lazy mmap loading — reads only first 4 channels.
    Track B = T1, T1c, T2, FLAIR = arr[s, :4, :, :]
    Reads directly from /kaggle/input/ — zero working directory usage.
    """
    def __init__(self, patient_ids, df_labels, slices_dir,
                 n_channels=4, augment=False):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.augment    = augment
        self.items      = []

        label_map = dict(zip(df_labels['patient_id'],
                             df_labels['IDH_binary'].astype(int)))
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s, label_map[pid]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s, label = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        # Only first 4 channels — Track B
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            if torch.rand(1) > 0.5:
                angle = (torch.rand(1).item() - 0.5) * 30
                x = TF.rotate(x, angle)
            scale = 0.9 + torch.rand(1).item() * 0.2
            x = x * scale
            x = x + torch.randn_like(x) * 0.02
        return x, label, pid

# ── MODEL ─────────────────────────────────────────────────────────────────────
def build_model(n_channels, n_classes):
    model    = resnet18(weights='IMAGENET1K_V1')
    old_conv = model.conv1
    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    with torch.no_grad():
        avg_weight = old_conv.weight.mean(dim=1, keepdim=True)
        new_conv.weight = nn.Parameter(
            avg_weight.repeat(1, n_channels, 1, 1)
        )
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, n_classes)
    )
    return model

# ── TRAINING ──────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

# ── EVALUATION ────────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device, label_map):
    model.eval()
    total_loss       = 0
    patient_probs    = {}
    all_slice_probs  = []
    all_slice_labels = []

    with torch.no_grad():
        for x, y, pids in loader:
            x, y   = x.to(device), y.to(device)
            out    = model(x)
            loss   = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            labels = y.cpu().numpy()
            all_slice_probs.extend(probs)
            all_slice_labels.extend(labels)
            for prob, pid in zip(probs, pids):
                patient_probs.setdefault(pid, []).append(float(prob))

    slice_auc      = roc_auc_score(all_slice_labels, all_slice_probs)
    pid_list       = list(patient_probs.keys())
    pid_mean_probs = [np.mean(patient_probs[p]) for p in pid_list]
    pid_labels     = [label_map[p] for p in pid_list]

    patient_auc = roc_auc_score(pid_labels, pid_mean_probs)
    pid_preds   = [1 if p >= 0.5 else 0 for p in pid_mean_probs]

    accuracy    = accuracy_score(pid_labels, pid_preds)
    sensitivity = recall_score(pid_labels, pid_preds, zero_division=0)
    f1          = f1_score(pid_labels, pid_preds, zero_division=0)
    brier       = brier_score_loss(pid_labels, pid_mean_probs)

    cm = confusion_matrix(pid_labels, pid_preds, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    metrics = {
        'patient_auc': round(float(patient_auc), 4),
        'slice_auc':   round(float(slice_auc), 4),
        'accuracy':    round(float(accuracy), 4),
        'sensitivity': round(float(sensitivity), 4),
        'specificity': round(float(specificity), 4),
        'f1':          round(float(f1), 4),
        'brier':       round(float(brier), 4),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
    }
    return total_loss / len(loader.dataset), metrics, patient_probs

# ── RELOAD PROGRESS ───────────────────────────────────────────────────────────
cnn_progress_path = f'{WORKING}/cnn_trackB_progress.json'

if os.path.exists(cnn_progress_path):
    with open(cnn_progress_path) as f:
        cnn_progress = json.load(f)
    completed_folds = cnn_progress.get('completed_folds', [])
    fold_results    = cnn_progress.get('fold_results', {})
    all_val_preds   = cnn_progress.get('all_val_preds', {})
    print(f'RESUMING CNN Track B — completed folds: {completed_folds}')
else:
    completed_folds = []
    fold_results    = {}
    all_val_preds   = {}
    print('Starting CNN Track B fresh')

# Restore Track B weights for completed folds
for fold in completed_folds:
    local_path   = f'{SAVE_DIR}/fold{fold}_best.pth'
    kaggle_name  = f'cnn_trackB_fold{fold}_best.pth'
    working_copy = os.path.join(WORKING, kaggle_name)
    if not os.path.exists(local_path) and os.path.exists(working_copy):
        shutil.copy(working_copy, local_path)
        print(f'Restored Track B fold {fold} weights')

# ── LOAD LABELS AND SPLITS ────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

label_map     = dict(zip(df_labels['patient_id'],
                         df_labels['IDH_binary'].astype(int)))
class_weights = torch.tensor([1.0, CLASS_WEIGHT],
                              dtype=torch.float32).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# ── 5-FOLD TRAINING LOOP ──────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK B — 5-FOLD CROSS VALIDATION')
print(f'{"="*60}')

for fold in range(N_FOLDS):
    if fold in completed_folds:
        print(f'\nFold {fold} — already done, skipping')
        continue

    print(f'\nFOLD {fold}/{N_FOLDS-1}')
    print('-' * 40)

    val_patients   = splits[f'fold_{fold}']
    train_patients = []
    for other_fold in range(N_FOLDS):
        if other_fold != fold:
            train_patients.extend(splits[f'fold_{other_fold}'])

    print(f'Train patients : {len(train_patients)}')
    print(f'Val patients   : {len(val_patients)}')

    train_ds = GliomaDataset(
        train_patients, df_labels,
        SLICES_DIR,
        n_channels=N_CHANNELS, augment=True
    )
    val_ds = GliomaDataset(
        val_patients, df_labels,
        SLICES_DIR,
        n_channels=N_CHANNELS, augment=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    print(f'Train slices   : {len(train_ds)}')
    print(f'Val slices     : {len(val_ds)}')

    model = build_model(N_CHANNELS, N_CLASSES).to(DEVICE)

    for name, param in model.named_parameters():
        if not name.startswith('fc'):
            param.requires_grad = False

    optimizer = optim.AdamW([
        {'params': model.fc.parameters(), 'lr': LR_HEAD},
        {'params': [p for n, p in model.named_parameters()
                    if not n.startswith('fc')],
         'lr': LR_BACKBONE}
    ], weight_decay=WEIGHT_DECAY)

    scheduler    = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    best_val_auc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    best_metrics = {}
    history      = []

    for epoch in range(1, MAX_EPOCHS + 1):

        if epoch == 6:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {'params': model.fc.parameters(), 'lr': LR_HEAD},
                {'params': [p for n, p in model.named_parameters()
                            if not n.startswith('fc')],
                 'lr': LR_BACKBONE}
            ], weight_decay=WEIGHT_DECAY)
            scheduler = CosineAnnealingLR(
                optimizer, T_max=MAX_EPOCHS - 5
            )
            print(f'  Epoch {epoch}: backbone unfrozen, optimizer reset')

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE
        )
        val_loss, metrics, _ = evaluate(
            model, val_loader, criterion, DEVICE, label_map
        )
        scheduler.step()

        history.append({
            'epoch':      epoch,
            'train_loss': round(float(train_loss), 4),
            'val_loss':   round(float(val_loss), 4),
            **metrics
        })

        print(
            f'  Epoch {epoch:02d} | '
            f'loss: {train_loss:.4f}/{val_loss:.4f} | '
            f'pt_AUC: {metrics["patient_auc"]:.4f} | '
            f'sl_AUC: {metrics["slice_auc"]:.4f} | '
            f'sens: {metrics["sensitivity"]:.4f} | '
            f'spec: {metrics["specificity"]:.4f} | '
            f'F1: {metrics["f1"]:.4f} | '
            f'Brier: {metrics["brier"]:.4f}'
        )

        if metrics['patient_auc'] > best_val_auc:
            best_val_auc = metrics['patient_auc']
            best_epoch   = epoch
            best_metrics = metrics.copy()
            patience_ctr = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/fold{fold}_best.pth')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(
                    f'  Early stopping at epoch {epoch} '
                    f'(best epoch {best_epoch}, '
                    f'best patient AUC {best_val_auc:.4f})'
                )
                break

    # Reload best weights for final evaluation
    print(f'  Reloading best weights (epoch {best_epoch})...')
    model.load_state_dict(
        torch.load(f'{SAVE_DIR}/fold{fold}_best.pth',
                   map_location=DEVICE)
    )
    _, best_metrics, patient_probs = evaluate(
        model, val_loader, criterion, DEVICE, label_map
    )

    for pid, probs in patient_probs.items():
        all_val_preds[pid] = {
            'mean_prob': float(np.mean(probs)),
            'label':     int(label_map[pid]),
            'fold':      fold
        }

    fold_results[str(fold)] = {
        'best_epoch':   best_epoch,
        'best_auc':     round(best_val_auc, 4),
        'best_metrics': best_metrics,
        'history':      history
    }
    completed_folds.append(fold)

    print(f'\n  Fold {fold} best results (epoch {best_epoch}):')
    print(f'    Patient AUC  : {best_metrics["patient_auc"]:.4f}')
    print(f'    Sensitivity  : {best_metrics["sensitivity"]:.4f}')
    print(f'    Specificity  : {best_metrics["specificity"]:.4f}')
    print(f'    F1 score     : {best_metrics["f1"]:.4f}')
    print(f'    Accuracy     : {best_metrics["accuracy"]:.4f}')
    print(f'    Brier score  : {best_metrics["brier"]:.4f}')
    print(f'    TP:{best_metrics["tp"]} TN:{best_metrics["tn"]} '
          f'FP:{best_metrics["fp"]} FN:{best_metrics["fn"]}')

    # Save progress locally
    with open(cnn_progress_path, 'w') as f:
        json.dump({
            'completed_folds': completed_folds,
            'fold_results':    fold_results,
            'all_val_preds':   all_val_preds
        }, f)

    # Copy weights to working root for commit
    shutil.copy(
        f'{SAVE_DIR}/fold{fold}_best.pth',
        f'{WORKING}/cnn_trackB_fold{fold}_best.pth'
    )

    # Safe cumulative commit
    safe_commit_to_kaggle(
        f'CNN Track B fold {fold} complete — AUC {best_val_auc:.4f}'
    )

    # Free GPU memory
    del model, optimizer, scheduler
    del train_ds, val_ds, train_loader, val_loader
    torch.cuda.empty_cache()

    # Check disk after each fold
    _, used, free = shutil.disk_usage(WORKING)
    print(f'  Disk: {used/1e9:.2f} GB used / {free/1e9:.2f} GB free')

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CNN TRACK B — CROSS VALIDATION COMPLETE')
print(f'{"="*60}')

aucs          = [fold_results[str(f)]['best_auc'] for f in range(N_FOLDS)]
sensitivities = [fold_results[str(f)]['best_metrics']['sensitivity']
                 for f in range(N_FOLDS)]
specificities = [fold_results[str(f)]['best_metrics']['specificity']
                 for f in range(N_FOLDS)]
f1s           = [fold_results[str(f)]['best_metrics']['f1']
                 for f in range(N_FOLDS)]
briers        = [fold_results[str(f)]['best_metrics']['brier']
                 for f in range(N_FOLDS)]
accuracies    = [fold_results[str(f)]['best_metrics']['accuracy']
                 for f in range(N_FOLDS)]

print(f'\nPer-fold AUC       : {[round(a, 4) for a in aucs]}')
print(f'Mean AUC           : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'Mean Sensitivity   : {np.mean(sensitivities):.4f} ± {np.std(sensitivities):.4f}')
print(f'Mean Specificity   : {np.mean(specificities):.4f} ± {np.std(specificities):.4f}')
print(f'Mean F1            : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'Mean Accuracy      : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}')
print(f'Mean Brier Score   : {np.mean(briers):.4f} ± {np.std(briers):.4f}')
print(f'\nPatients with predictions : {len(all_val_preds)} (expect 495)')
print(f'All weights saved to      : {MODELS_DATASET}')
print(f'\nNext: Cell 7 — Radiomics feature extraction')
print(f'\nNote: Wilcoxon signed-rank tests comparing Track A vs Track B')
print(f'will be run in Cell 13 when all models are complete.')

Kaggle API ready — saving to adesaladaniel/glioma-idh-models
Device     : cuda
Track      : B — 4 channels (T1, T1c, T2, FLAIR)
Class wt   : 3.81
Slices dir : /kaggle/input/datasets/adesaladaniel/glioma-idh-tensors

Syncing existing model files from Kaggle...
Downloaded 11 existing files:
  ✅ cnn_trackA_fold0_best.pth
  ✅ cnn_trackA_fold1_best.pth
  ✅ cnn_trackA_fold2_best.pth
  ✅ cnn_trackA_fold3_best.pth
  ✅ cnn_trackA_fold4_best.pth
  ✅ cnn_trackA_progress.json
  ✅ cnn_trackB_fold0_best.pth
  ✅ cnn_trackB_fold1_best.pth
  ✅ cnn_trackB_fold2_best.pth
  ✅ cnn_trackB_fold3_best.pth
  ✅ cnn_trackB_progress.json
Sync complete.

RESUMING CNN Track B — completed folds: [0, 1, 2, 3]

CNN TRACK B — 5-FOLD CROSS VALIDATION

Fold 0 — already done, skipping

Fold 1 — already done, skipping

Fold 2 — already done, skipping

Fold 3 — already done, skipping

FOLD 4/4
----------------------------------------
Train patients : 396
Val patients   : 99
Train slices   : 23199
Val slices     : 5682
  Epo

## Cell 7: Radiomics feature extraction



In [ ]:
# ── CELL 7: RADIOMICS FEATURE EXTRACTION ─────────────────────────────────────
# Extracts PyRadiomics features from original NIfTI files
# Track A: ~900 features (7 sequences × feature classes + shape)
# Track B: ~550 features (4 sequences × feature classes + shape)
#
# Install : PyRadiomics via git (Python 3.12 compatible)
# Reads   : original NIfTI batch folders
# Saves   : radiomics_trackA.csv + radiomics_trackB.csv
# Commits : to adesaladaniel/glioma-idh-models
# Resume  : safe — skips completed patients automatically
# Time    : ~3-4 hours CPU

import os, sys, subprocess

# ── INSTALL PYRADIOMICS ───────────────────────────────────────────────────────
print('Installing PyRadiomics (git)...')
os.system('pip install "pyradiomics @ git+https://github.com/'
          'AIM-Harvard/pyradiomics.git" -q')

try:
    from radiomics import featureextractor
    import radiomics
    print(f'✅ PyRadiomics {radiomics.__version__} ready')
except ImportError as e:
    print(f'❌ PyRadiomics import failed: {e}')
    raise

import os, json, shutil, warnings
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import logging

warnings.filterwarnings('ignore')
logging.getLogger('radiomics').setLevel(logging.ERROR)
logging.getLogger('pykwalify').setLevel(logging.ERROR)

WORKING     = '/kaggle/working'
KAGGLE_USER = 'adesaladaniel'
BATCH_ROOTS = [
    f'/kaggle/input/datasets/{KAGGLE_USER}/ucsf-pdgm-batch-{i:02d}'
    for i in range(1, 11)
]

TRACK_A_SEQS   = ['T1', 'T1c', 'T2', 'FLAIR', 'ADC', 'DTI_eddy_FA', 'DTI_eddy_MD']
TRACK_B_SEQS   = ['T1', 'T1c', 'T2', 'FLAIR']
TUMOR_SEQ      = 'tumor_segmentation'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

# ── REBUILD PATIENT DIRS ──────────────────────────────────────────────────────
patient_dirs = {}
for batch_root in BATCH_ROOTS:
    if not os.path.exists(batch_root):
        continue
    for entry in sorted(os.listdir(batch_root)):
        if entry.startswith('UCSF-PDGM-'):
            full_path = os.path.join(batch_root, entry)
            if os.path.isdir(full_path):
                patient_dirs[entry] = full_path

print(f'Patient folders : {len(patient_dirs)}/495')

# ── LOAD LABELS ───────────────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))
fold_map  = dict(zip(df_labels['patient_id'],
                     df_labels['fold'].astype(int)))
print(f'Labels loaded   : {len(df_labels)} patients')

# ── NIfTI PATH HELPER ─────────────────────────────────────────────────────────
def get_nii_path(patient_dir, patient_id, seq_name):
    filename = f'{patient_id}_{seq_name}.nii'
    nested   = os.path.join(patient_dir, filename, filename)
    flat     = os.path.join(patient_dir, filename)
    if os.path.isfile(nested):
        return nested
    elif os.path.isfile(flat):
        return flat
    return None

# ── PYRADIOMICS EXTRACTOR ─────────────────────────────────────────────────────
def build_extractor():
    params = {
        'imageType': {
            'Original': {}
        },
        'featureClass': {
            'firstorder': [],
            'shape':      [],
            'glcm':       [],
            'glrlm':      [],
            'glszm':      []
        },
        'setting': {
            'binWidth':              25,
            'resampledPixelSpacing': [1, 1, 1],
            'interpolator':          'sitkBSpline',
            'padDistance':           10,
            'normalize':             True,
            'normalizeScale':        100,
            'removeOutliers':        3.0,
            'minimumROISize':        10,
            'geometryTolerance':     1e-6,
            'correctMask':           True
        }
    }
    return featureextractor.RadiomicsFeatureExtractor(params)

extractor = build_extractor()
print('Extractor configured ✅')

# ── FEATURE EXTRACTION FUNCTION ───────────────────────────────────────────────
def extract_features_for_patient(patient_id, patient_dir, sequences):
    try:
        # Load tumor mask
        tumor_path = get_nii_path(patient_dir, patient_id, TUMOR_SEQ)
        if tumor_path is None:
            return None, 'No tumor mask found'

        mask_sitk_orig = sitk.ReadImage(tumor_path)
        mask_arr       = sitk.GetArrayFromImage(mask_sitk_orig)
        mask_bin       = (mask_arr > 0).astype(np.uint8)

        if mask_bin.sum() < 10:
            return None, f'Tumor mask too small ({mask_bin.sum()} voxels)'

        mask_sitk = sitk.GetImageFromArray(mask_bin)
        mask_sitk.CopyInformation(mask_sitk_orig)

        all_features = {}

        for seq in sequences:
            seq_path = get_nii_path(patient_dir, patient_id, seq)
            if seq_path is None:
                return None, f'Missing sequence: {seq}'

            img_sitk = sitk.ReadImage(seq_path)
            img_sitk = sitk.Cast(img_sitk, sitk.sitkFloat32)

            result = extractor.execute(img_sitk, mask_sitk, label=1)

            for key, val in result.items():
                if key.startswith('original_'):
                    clean_key = key.replace('original_', f'{seq}_')
                    try:
                        all_features[clean_key] = float(val)
                    except (TypeError, ValueError):
                        pass

        return all_features, None

    except Exception as e:
        return None, str(e)

# ── LOAD OR INITIALISE PROGRESS ───────────────────────────────────────────────
progress_path_A = f'{WORKING}/radiomics_trackA_progress.json'
progress_path_B = f'{WORKING}/radiomics_trackB_progress.json'

# ── CHECK KAGGLE FOR EXISTING PROGRESS ───────────────────────────────────────
print('\nChecking Kaggle for existing radiomics progress...')
sync_dir = f'{WORKING}/rad_sync'
os.makedirs(sync_dir, exist_ok=True)

sync_result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

if sync_result.returncode == 0:
    for fname in ['radiomics_trackA_progress.json',
                  'radiomics_trackB_progress.json',
                  'radiomics_trackA.csv',
                  'radiomics_trackB.csv']:
        src = os.path.join(sync_dir, fname)
        dst = os.path.join(WORKING, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Restored: {fname}')

shutil.rmtree(sync_dir, ignore_errors=True)

# Track A progress
if os.path.exists(progress_path_A):
    with open(progress_path_A) as f:
        prog_A = json.load(f)
    completed_A = set(prog_A.get('completed', []))
    failed_A    = prog_A.get('failed', {})
    rows_A      = prog_A.get('rows', [])
    print(f'Track A: resuming — {len(completed_A)} done, '
          f'{len(failed_A)} failed')
else:
    completed_A = set()
    failed_A    = {}
    rows_A      = []
    print('Track A: starting fresh')

# Track B progress
if os.path.exists(progress_path_B):
    with open(progress_path_B) as f:
        prog_B = json.load(f)
    completed_B = set(prog_B.get('completed', []))
    failed_B    = prog_B.get('failed', {})
    rows_B      = prog_B.get('rows', [])
    print(f'Track B: resuming — {len(completed_B)} done, '
          f'{len(failed_B)} failed')
else:
    completed_B = set()
    failed_B    = {}
    rows_B      = []
    print('Track B: starting fresh')

# ── MAIN EXTRACTION LOOP ──────────────────────────────────────────────────────
patient_list = df_labels['patient_id'].tolist()
n_total      = len(patient_list)

print(f'\n{"="*60}')
print('RADIOMICS EXTRACTION')
print(f'{"="*60}')
print(f'Total patients    : {n_total}')
print(f'Track A remaining : {n_total - len(completed_A)}')
print(f'Track B remaining : {n_total - len(completed_B)}')
print()

for i, patient_id in enumerate(patient_list):
    patient_dir = patient_dirs.get(patient_id)
    if patient_dir is None:
        continue

    idh_label = label_map[patient_id]
    fold      = fold_map[patient_id]

    # ── TRACK A ───────────────────────────────────────────────────────────────
    if patient_id not in completed_A:
        feats_A, err_A = extract_features_for_patient(
            patient_id, patient_dir, TRACK_A_SEQS)

        if err_A:
            failed_A[patient_id] = err_A
        else:
            row = {'patient_id': patient_id,
                   'IDH_binary': idh_label,
                   'fold':       fold}
            row.update(feats_A)
            rows_A.append(row)
            completed_A.add(patient_id)

    # ── TRACK B ───────────────────────────────────────────────────────────────
    if patient_id not in completed_B:
        feats_B, err_B = extract_features_for_patient(
            patient_id, patient_dir, TRACK_B_SEQS)

        if err_B:
            failed_B[patient_id] = err_B
        else:
            row = {'patient_id': patient_id,
                   'IDH_binary': idh_label,
                   'fold':       fold}
            row.update(feats_B)
            rows_B.append(row)
            completed_B.add(patient_id)

    # ── PROGRESS PRINT ────────────────────────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        print(f'  [{i+1:>3}/{n_total}] '
              f'A: {len(completed_A):>3} done / {len(failed_A):>2} failed | '
              f'B: {len(completed_B):>3} done / {len(failed_B):>2} failed')

    # ── SAVE PROGRESS EVERY 10 PATIENTS ──────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == n_total:
        with open(progress_path_A, 'w') as f:
            json.dump({'completed': list(completed_A),
                       'failed':    failed_A,
                       'rows':      rows_A}, f)
        with open(progress_path_B, 'w') as f:
            json.dump({'completed': list(completed_B),
                       'failed':    failed_B,
                       'rows':      rows_B}, f)

    # ── COMMIT TO KAGGLE EVERY 50 PATIENTS ───────────────────────────────────
    if (i + 1) % 50 == 0:
        print(f'  Committing progress to Kaggle...')
        commit_dir = f'{WORKING}/rad_commit'
        os.makedirs(commit_dir, exist_ok=True)

        with open(os.path.join(commit_dir,
                               'dataset-metadata.json'), 'w') as f:
            json.dump({'title':     'Glioma IDH Models',
                       'id':        MODELS_DATASET,
                       'licenses':  [{'name': 'other'}],
                       'isPrivate': True}, f)

        for fname in os.listdir(WORKING):
            src = os.path.join(WORKING, fname)
            lnk = os.path.join(commit_dir, fname)
            if (fname.startswith('cnn_track') or
                    fname.startswith('radiomics_')) and \
               (fname.endswith('.pth') or
                fname.endswith('.json') or
                fname.endswith('.csv')):
                if os.path.exists(src) and not os.path.lexists(lnk):
                    os.symlink(os.path.abspath(src), lnk)

        r = subprocess.run(
            ['kaggle', 'datasets', 'version',
             '-p', commit_dir,
             '-m', f'Radiomics progress: '
                   f'A={len(completed_A)} B={len(completed_B)}',
             '--dir-mode', 'zip'],
            capture_output=True, text=True
        )
        status = '✅' if r.returncode == 0 else '⚠️'
        print(f'  {status} Kaggle commit '
              f'(A={len(completed_A)}, B={len(completed_B)})')
        shutil.rmtree(commit_dir, ignore_errors=True)

# ── SAVE FINAL CSVs ───────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('SAVING FINAL CSVs')
print(f'{"="*60}')

df_A = pd.DataFrame(rows_A).sort_values(
    'patient_id').reset_index(drop=True)
df_B = pd.DataFrame(rows_B).sort_values(
    'patient_id').reset_index(drop=True)

csv_path_A = f'{WORKING}/radiomics_trackA.csv'
csv_path_B = f'{WORKING}/radiomics_trackB.csv'

df_A.to_csv(csv_path_A, index=False)
df_B.to_csv(csv_path_B, index=False)

print(f'Track A : {len(df_A)} patients × '
      f'{len(df_A.columns)-3} features')
print(f'Track B : {len(df_B)} patients × '
      f'{len(df_B.columns)-3} features')

if failed_A:
    print(f'\nTrack A failed ({len(failed_A)}):')
    for pid, reason in list(failed_A.items())[:5]:
        print(f'  {pid}: {reason}')
if failed_B:
    print(f'\nTrack B failed ({len(failed_B)}):')
    for pid, reason in list(failed_B.items())[:5]:
        print(f'  {pid}: {reason}')

# ── FINAL KAGGLE COMMIT ───────────────────────────────────────────────────────
print(f'\nFinal commit to Kaggle...')
commit_dir = f'{WORKING}/rad_commit_final'
os.makedirs(commit_dir, exist_ok=True)

with open(os.path.join(commit_dir, 'dataset-metadata.json'), 'w') as f:
    json.dump({'title':     'Glioma IDH Models',
               'id':        MODELS_DATASET,
               'licenses':  [{'name': 'other'}],
               'isPrivate': True}, f)

for fname in os.listdir(WORKING):
    src = os.path.join(WORKING, fname)
    lnk = os.path.join(commit_dir, fname)
    if (fname.startswith('cnn_track') or
            fname.startswith('radiomics_')) and \
       (fname.endswith('.pth') or
        fname.endswith('.json') or
        fname.endswith('.csv')):
        if os.path.exists(src) and not os.path.lexists(lnk):
            os.symlink(os.path.abspath(src), lnk)

r = subprocess.run(
    ['kaggle', 'datasets', 'version',
     '-p', commit_dir,
     '-m', f'Radiomics complete — '
           f'A:{len(df_A)} B:{len(df_B)} patients',
     '--dir-mode', 'zip'],
    capture_output=True, text=True
)

if r.returncode == 0:
    print('✅ Radiomics CSVs permanently saved to Kaggle')
else:
    print(f'⚠️  Commit failed: {r.stderr[:200]}')
    print('CSVs are in /kaggle/working/ — safe until session ends')

shutil.rmtree(commit_dir, ignore_errors=True)

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('CELL 7 COMPLETE')
print(f'{"="*60}')
print(f'Track A : {len(df_A)}/495 patients, '
      f'{len(df_A.columns)-3} features')
print(f'Track B : {len(df_B)}/495 patients, '
      f'{len(df_B.columns)-3} features')
print(f'Failed  : A={len(failed_A)}, B={len(failed_B)}')
print()
print('Improvements over basic Cell 7:')
print('  ✅ Commits to Kaggle every 50 patients (resume-safe)')
print('  ✅ Restores existing progress from Kaggle on startup')
print('  ✅ Both tracks extracted in single pass')
print()
print('Next: Cell 8 — Feature selection + SVM/XGBoost + SHAP')

Installing PyRadiomics (git)...
✅ PyRadiomics 3.1.1.dev111+g8ed579383 ready
Patient folders : 495/495
Labels loaded   : 495 patients
Extractor configured ✅

Checking Kaggle for existing radiomics progress...
  Restored: radiomics_trackA_progress.json
  Restored: radiomics_trackB_progress.json
Track A: resuming — 400 done, 0 failed
Track B: resuming — 400 done, 0 failed

RADIOMICS EXTRACTION
Total patients    : 495
Track A remaining : 95
Track B remaining : 95

  [ 10/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 20/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 30/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 40/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 50/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  Committing progress to Kaggle...
  ✅ Kaggle commit (A=400, B=400)
  [ 60/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 70/495] A: 400 done /  0 failed | B: 400 done /  0 failed
  [ 80/495] A: 400 done /  0 fail

## Cell 8: RADIOMICS CLASSIFICATION + SHAP


In [ ]:
# ── CELL 8: RADIOMICS CLASSIFICATION + SHAP ───────────────────────────────────

# Feature selection pipeline per fold:
#   VarianceThreshold → LassoCV → 20-78 features
# Classifiers: SVM + XGBoost (both tested, best reported)
# SHAP: explains which features matter most
# Saves: radiomics_trackA_results.json
#        radiomics_trackB_results.json
#        radiomics_trackA_shap.csv
#        radiomics_trackB_shap.csv

import os, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

WORKING        = '/kaggle/working'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

# ── STEP 1: CHECK IF RESULTS ALREADY EXIST ───────────────────────────────────
print('Checking for existing Cell 8 results...')

# Download from Kaggle first
sync_dir = f'{WORKING}/cell8_sync'
os.makedirs(sync_dir, exist_ok=True)

subprocess.run(
    ['kaggle', 'datasets', 'download',
     MODELS_DATASET, '--path', sync_dir, '--unzip'],
    capture_output=True, text=True
)

# Copy relevant files to working
for fname in os.listdir(sync_dir):
    src = os.path.join(sync_dir, fname)
    dst = os.path.join(WORKING, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

shutil.rmtree(sync_dir, ignore_errors=True)

results_A_path = f'{WORKING}/radiomics_trackA_results.json'
results_B_path = f'{WORKING}/radiomics_trackB_results.json'

# ── STEP 2: IF RESULTS EXIST — DISPLAY AND STOP ──────────────────────────────
if os.path.exists(results_A_path) and \
   os.path.exists(results_B_path):

    print('✅ Cell 8 results found — loading...\n')

    with open(results_A_path) as f:
        res_A = json.load(f)
    with open(results_B_path) as f:
        res_B = json.load(f)

    # ── DISPLAY TRACK A ───────────────────────────────────────
    print('='*60)
    print('RADIOMICS TRACK A — RESULTS')
    print('='*60)

    svm_aucs_A, xgb_aucs_A = [], []
    svm_sens_A, xgb_sens_A = [], []
    svm_spec_A, xgb_spec_A = [], []

    for fold in range(5):
        if str(fold) in res_A['fold_results']:
            r   = res_A['fold_results'][str(fold)]
            svm = r['svm_metrics']
            xgb = r['xgb_metrics']
            svm_aucs_A.append(svm['patient_auc'])
            xgb_aucs_A.append(xgb['patient_auc'])
            svm_sens_A.append(svm['sensitivity'])
            xgb_sens_A.append(xgb['sensitivity'])
            svm_spec_A.append(svm['specificity'])
            xgb_spec_A.append(xgb['specificity'])
            print(f'\nFold {fold} '
                  f'({r["n_features"]} features selected):')
            print(f'  SVM     — '
                  f'AUC: {svm["patient_auc"]:.4f} | '
                  f'Sens: {svm["sensitivity"]:.4f} | '
                  f'Spec: {svm["specificity"]:.4f} | '
                  f'F1: {svm["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb["patient_auc"]:.4f} | '
                  f'Sens: {xgb["sensitivity"]:.4f} | '
                  f'Spec: {xgb["specificity"]:.4f} | '
                  f'F1: {xgb["f1"]:.4f}')
            print(f'  Best    : {r["best"]}')

    print(f'\nSVM Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in svm_aucs_A]}')
    print(f'  Mean AUC     : '
          f'{np.mean(svm_aucs_A):.4f} ± '
          f'{np.std(svm_aucs_A):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(svm_sens_A):.4f} ± '
          f'{np.std(svm_sens_A):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(svm_spec_A):.4f} ± '
          f'{np.std(svm_spec_A):.4f}')

    print(f'\nXGBoost Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in xgb_aucs_A]}')
    print(f'  Mean AUC     : '
          f'{np.mean(xgb_aucs_A):.4f} ± '
          f'{np.std(xgb_aucs_A):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(xgb_sens_A):.4f} ± '
          f'{np.std(xgb_sens_A):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(xgb_spec_A):.4f} ± '
          f'{np.std(xgb_spec_A):.4f}')

    # Feature consistency Track A
    feature_counts_A = {}
    for f in range(5):
        if str(f) in res_A['fold_results']:
            for fname in res_A['fold_results'][str(f)][
                    'feature_names']:
                feature_counts_A[fname] = \
                    feature_counts_A.get(fname, 0) + 1

    print(f'\nTop 10 most consistently selected features:')
    for fname, count in sorted(
            feature_counts_A.items(),
            key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {count}/5 folds — {fname}')

    # ── DISPLAY TRACK B ───────────────────────────────────────
    print(f'\n{"="*60}')
    print('RADIOMICS TRACK B — RESULTS')
    print('='*60)

    svm_aucs_B, xgb_aucs_B = [], []
    svm_sens_B, xgb_sens_B = [], []
    svm_spec_B, xgb_spec_B = [], []

    for fold in range(5):
        if str(fold) in res_B['fold_results']:
            r   = res_B['fold_results'][str(fold)]
            svm = r['svm_metrics']
            xgb = r['xgb_metrics']
            svm_aucs_B.append(svm['patient_auc'])
            xgb_aucs_B.append(xgb['patient_auc'])
            svm_sens_B.append(svm['sensitivity'])
            xgb_sens_B.append(xgb['sensitivity'])
            svm_spec_B.append(svm['specificity'])
            xgb_spec_B.append(xgb['specificity'])
            print(f'\nFold {fold} '
                  f'({r["n_features"]} features selected):')
            print(f'  SVM     — '
                  f'AUC: {svm["patient_auc"]:.4f} | '
                  f'Sens: {svm["sensitivity"]:.4f} | '
                  f'Spec: {svm["specificity"]:.4f} | '
                  f'F1: {svm["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb["patient_auc"]:.4f} | '
                  f'Sens: {xgb["sensitivity"]:.4f} | '
                  f'Spec: {xgb["specificity"]:.4f} | '
                  f'F1: {xgb["f1"]:.4f}')
            print(f'  Best    : {r["best"]}')

    print(f'\nSVM Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in svm_aucs_B]}')
    print(f'  Mean AUC     : '
          f'{np.mean(svm_aucs_B):.4f} ± '
          f'{np.std(svm_aucs_B):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(svm_sens_B):.4f} ± '
          f'{np.std(svm_sens_B):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(svm_spec_B):.4f} ± '
          f'{np.std(svm_spec_B):.4f}')

    print(f'\nXGBoost Results:')
    print(f'  Per-fold AUC : '
          f'{[round(a,4) for a in xgb_aucs_B]}')
    print(f'  Mean AUC     : '
          f'{np.mean(xgb_aucs_B):.4f} ± '
          f'{np.std(xgb_aucs_B):.4f}')
    print(f'  Mean Sens    : '
          f'{np.mean(xgb_sens_B):.4f} ± '
          f'{np.std(xgb_sens_B):.4f}')
    print(f'  Mean Spec    : '
          f'{np.mean(xgb_spec_B):.4f} ± '
          f'{np.std(xgb_spec_B):.4f}')

    # Feature consistency Track B
    feature_counts_B = {}
    for f in range(5):
        if str(f) in res_B['fold_results']:
            for fname in res_B['fold_results'][str(f)][
                    'feature_names']:
                feature_counts_B[fname] = \
                    feature_counts_B.get(fname, 0) + 1

    print(f'\nTop 10 most consistently selected features:')
    for fname, count in sorted(
            feature_counts_B.items(),
            key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {count}/5 folds — {fname}')

    # ── FINAL COMPARISON ──────────────────────────────────────
    best_A = max(np.mean(svm_aucs_A), np.mean(xgb_aucs_A))
    best_B = max(np.mean(svm_aucs_B), np.mean(xgb_aucs_B))
    clf_A  = ('SVM' if np.mean(svm_aucs_A) >= np.mean(xgb_aucs_A)
              else 'XGBoost')
    clf_B  = ('SVM' if np.mean(svm_aucs_B) >= np.mean(xgb_aucs_B)
              else 'XGBoost')
    std_A  = (np.std(svm_aucs_A) if clf_A == 'SVM'
              else np.std(xgb_aucs_A))
    std_B  = (np.std(svm_aucs_B) if clf_B == 'SVM'
              else np.std(xgb_aucs_B))

    print(f'\n{"="*60}')
    print('RADIOMICS vs CNN — COMPARISON')
    print(f'{"="*60}')
    print(f'\n{"Model":<30} {"AUC":>8} {"±":>4} {"STD":>8}')
    print('-'*52)
    print(f'{"CNN Track A (ResNet18)":<30} '
          f'{"0.9654":>8} {"±":>4} {"0.0190":>8}')
    print(f'{"Radiomics Track A ("+clf_A+")":<30} '
          f'{best_A:>8.4f} {"±":>4} {std_A:>8.4f}')
    print()
    print(f'{"CNN Track B (ResNet18)":<30} '
          f'{"0.9636":>8} {"±":>4} {"0.0129":>8}')
    print(f'{"Radiomics Track B ("+clf_B+")":<30} '
          f'{best_B:>8.4f} {"±":>4} {std_B:>8.4f}')
    print()
    print(f'CNN vs Radiomics delta Track A : '
          f'{0.9654 - best_A:+.4f}')
    print(f'CNN vs Radiomics delta Track B : '
          f'{0.9636 - best_B:+.4f}')
    print(f'\n✅ Cell 8 complete — results loaded from Kaggle')
    print(f'✅ No retraining needed')
    print(f'Next: Cell 9 — Hybrid Fusion embeddings')

# ── STEP 3: IF NO RESULTS — RUN FULL PIPELINE ────────────────────────────────
else:
    print('❌ No existing results found')
    print('Running full Cell 8 classification pipeline...\n')

    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.linear_model import LassoCV
    from sklearn.svm import SVC
    from sklearn.metrics import (
        roc_auc_score, accuracy_score, recall_score,
        f1_score, confusion_matrix, brier_score_loss
    )
    import xgboost as xgb
    import shap

    N_FOLDS      = 5
    CLASS_WEIGHT = 3.81
    SEED         = 42
    META_COLS    = ['patient_id', 'IDH_binary', 'fold']

    df_A = pd.read_csv(f'{WORKING}/radiomics_trackA.csv')
    df_B = pd.read_csv(f'{WORKING}/radiomics_trackB.csv')

    feature_cols_A = [c for c in df_A.columns
                      if c not in META_COLS]
    feature_cols_B = [c for c in df_B.columns
                      if c not in META_COLS]

    print(f'Track A: {len(df_A)} patients × '
          f'{len(feature_cols_A)} features')
    print(f'Track B: {len(df_B)} patients × '
          f'{len(feature_cols_B)} features')

    def compute_metrics(y_true, y_pred, y_prob):
        cm = confusion_matrix(y_true, y_pred, labels=[0,1])
        tn, fp, fn, tp = (cm.ravel() if cm.shape==(2,2)
                          else (0,0,0,0))
        return {
            'patient_auc': round(float(
                roc_auc_score(y_true, y_prob)), 4),
            'accuracy':    round(float(
                accuracy_score(y_true, y_pred)), 4),
            'sensitivity': round(float(
                recall_score(y_true, y_pred,
                             zero_division=0)), 4),
            'specificity': round(float(
                tn/(tn+fp) if (tn+fp)>0 else 0.0), 4),
            'f1':          round(float(
                f1_score(y_true, y_pred,
                         zero_division=0)), 4),
            'brier':       round(float(
                brier_score_loss(y_true, y_prob)), 4),
            'tp': int(tp), 'tn': int(tn),
            'fp': int(fp), 'fn': int(fn)
        }

    def select_features(X_train, y_train, feature_names):
        selected = list(feature_names)
        X_work   = X_train.copy()
        vt       = VarianceThreshold(threshold=0.01)
        vt.fit(X_work)
        mask_vt  = vt.get_support()
        X_work   = X_work[:, mask_vt]
        selected = [n for n, m in zip(selected, mask_vt) if m]
        print(f'    After VarianceThreshold : '
              f'{len(selected)} features')
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(X_work)
        lasso    = LassoCV(cv=3, max_iter=5000,
                           random_state=SEED, n_jobs=-1)
        lasso.fit(X_scaled, y_train)
        mask_lasso = np.abs(lasso.coef_) > 1e-6
        selected   = [n for n, m in zip(selected, mask_lasso)
                      if m]
        print(f'    After LassoCV           : '
              f'{len(selected)} features')
        return selected, vt, scaler

    def commit_to_kaggle(message):
        commit_dir = f'{WORKING}/rad8_commit'
        os.makedirs(commit_dir, exist_ok=True)
        with open(os.path.join(commit_dir,
                               'dataset-metadata.json'), 'w') as f:
            json.dump({'title':     'Glioma IDH Models',
                       'id':        MODELS_DATASET,
                       'licenses':  [{'name': 'other'}],
                       'isPrivate': True}, f)
        for fname in os.listdir(WORKING):
            src = os.path.join(WORKING, fname)
            lnk = os.path.join(commit_dir, fname)
            if any(fname.startswith(p) for p in
                   ['cnn_track', 'radiomics_']) and \
               any(fname.endswith(e) for e in
                   ['.pth', '.json', '.csv']):
                if os.path.exists(src) and \
                   not os.path.lexists(lnk):
                    os.symlink(os.path.abspath(src), lnk)
        r = subprocess.run(
            ['kaggle', 'datasets', 'version',
             '-p', commit_dir, '-m', message,
             '--dir-mode', 'zip'],
            capture_output=True, text=True)
        status = '✅' if r.returncode == 0 else '⚠️'
        print(f'  {status} Kaggle commit: {message}')
        shutil.rmtree(commit_dir, ignore_errors=True)

    def run_radiomics_cv(df, feature_cols, track_name):
        print(f'\n{"="*60}')
        print(f'RADIOMICS {track_name} — 5-FOLD CV')
        print(f'{"="*60}')

        all_val_preds   = {}
        fold_results    = {}
        shap_accumulate = []

        for fold in range(N_FOLDS):
            print(f'\nFold {fold}')
            print('-'*40)

            train_df = df[df['fold'] != fold].copy()
            val_df   = df[df['fold'] == fold].copy()

            X_train  = train_df[feature_cols].values.astype(
                np.float32)
            y_train  = train_df['IDH_binary'].values.astype(int)
            X_val    = val_df[feature_cols].values.astype(
                np.float32)
            y_val    = val_df['IDH_binary'].values.astype(int)
            pids_val = val_df['patient_id'].values

            print(f'  Train: {len(train_df)} '
                  f'({(y_train==0).sum()} WT / '
                  f'{(y_train==1).sum()} MT)')
            print(f'  Val  : {len(val_df)} '
                  f'({(y_val==0).sum()} WT / '
                  f'{(y_val==1).sum()} MT)')
            print(f'  Feature selection:')

            selected_names, vt, scaler = select_features(
                X_train, y_train, feature_cols)

            if len(selected_names) == 0:
                print(f'  WARNING: No features selected')
                continue

            mask_vt     = vt.get_support()
            X_tr_vt     = X_train[:, mask_vt]
            X_vl_vt     = X_val[:, mask_vt]
            X_tr_scaled = scaler.transform(X_tr_vt)
            X_vl_scaled = scaler.transform(X_vl_vt)

            all_after_vt  = [f for f, m in
                             zip(feature_cols, mask_vt) if m]
            lasso_indices = [i for i, n in
                             enumerate(all_after_vt)
                             if n in selected_names]

            X_tr_sel = X_tr_scaled[:, lasso_indices]
            X_vl_sel = X_vl_scaled[:, lasso_indices]
            print(f'  Final features: {X_tr_sel.shape[1]}')

            svm = SVC(kernel='rbf', probability=True,
                      class_weight={0:1.0, 1:CLASS_WEIGHT},
                      C=1.0, gamma='scale', random_state=SEED)
            svm.fit(X_tr_sel, y_train)
            svm_probs   = svm.predict_proba(X_vl_sel)[:, 1]
            svm_preds   = (svm_probs >= 0.5).astype(int)
            svm_metrics = compute_metrics(
                y_val, svm_preds, svm_probs)

            scale_pos = ((y_train==0).sum() /
                         (y_train==1).sum())
            xgb_model = xgb.XGBClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos,
                eval_metric='logloss',
                random_state=SEED, n_jobs=-1)
            xgb_model.fit(X_tr_sel, y_train,
                          eval_set=[(X_vl_sel, y_val)],
                          verbose=False)
            xgb_probs   = xgb_model.predict_proba(X_vl_sel)[:, 1]
            xgb_preds   = (xgb_probs >= 0.5).astype(int)
            xgb_metrics = compute_metrics(
                y_val, xgb_preds, xgb_probs)

            print(f'  SVM     — '
                  f'AUC: {svm_metrics["patient_auc"]:.4f} | '
                  f'Sens: {svm_metrics["sensitivity"]:.4f} | '
                  f'Spec: {svm_metrics["specificity"]:.4f} | '
                  f'F1: {svm_metrics["f1"]:.4f}')
            print(f'  XGBoost — '
                  f'AUC: {xgb_metrics["patient_auc"]:.4f} | '
                  f'Sens: {xgb_metrics["sensitivity"]:.4f} | '
                  f'Spec: {xgb_metrics["specificity"]:.4f} | '
                  f'F1: {xgb_metrics["f1"]:.4f}')

            if xgb_metrics['patient_auc'] >= \
               svm_metrics['patient_auc']:
                best_probs = xgb_probs
                best_name  = 'XGBoost'
                best_model = xgb_model
            else:
                best_probs = svm_probs
                best_name  = 'SVM'
                best_model = svm

            print(f'  Best: {best_name}')

            label_map_fold = dict(zip(
                val_df['patient_id'], val_df['IDH_binary']))
            for pid, prob in zip(pids_val, best_probs):
                all_val_preds[pid] = {
                    'mean_prob': float(prob),
                    'label':     int(label_map_fold[pid]),
                    'fold':      fold
                }

            fold_results[str(fold)] = {
                'svm_metrics':   svm_metrics,
                'xgb_metrics':   xgb_metrics,
                'best':          best_name,
                'n_features':    X_tr_sel.shape[1],
                'feature_names': selected_names
            }

            try:
                explainer = shap.TreeExplainer(xgb_model)
                shap_vals = explainer.shap_values(X_vl_sel)
                for j, pid in enumerate(pids_val):
                    for k, fname in enumerate(selected_names):
                        shap_accumulate.append({
                            'patient_id':    pid,
                            'fold':          fold,
                            'feature':       fname,
                            'shap_value':    float(
                                shap_vals[j, k]),
                            'feature_value': float(
                                X_vl_sel[j, k])
                        })
            except Exception as e:
                print(f'  SHAP skipped: {e}')

        # Summary
        svm_aucs = [fold_results[str(f)]['svm_metrics'][
            'patient_auc'] for f in range(N_FOLDS)
            if str(f) in fold_results]
        xgb_aucs = [fold_results[str(f)]['xgb_metrics'][
            'patient_auc'] for f in range(N_FOLDS)
            if str(f) in fold_results]

        print(f'\n{"="*60}')
        print(f'RADIOMICS {track_name} — SUMMARY')
        print(f'{"="*60}')
        print(f'SVM     Mean AUC: '
              f'{np.mean(svm_aucs):.4f} ± '
              f'{np.std(svm_aucs):.4f}')
        print(f'XGBoost Mean AUC: '
              f'{np.mean(xgb_aucs):.4f} ± '
              f'{np.std(xgb_aucs):.4f}')

        feature_counts = {}
        for f in range(N_FOLDS):
            if str(f) in fold_results:
                for fname in fold_results[str(f)][
                        'feature_names']:
                    feature_counts[fname] = \
                        feature_counts.get(fname, 0) + 1

        print(f'\nTop 10 consistently selected features:')
        for fname, count in sorted(
                feature_counts.items(),
                key=lambda x: x[1], reverse=True)[:10]:
            print(f'  {count}/5 folds — {fname}')

        return fold_results, all_val_preds, shap_accumulate

    results_A, preds_A, shap_A = run_radiomics_cv(
        df_A, feature_cols_A, 'TRACK A')
    results_B, preds_B, shap_B = run_radiomics_cv(
        df_B, feature_cols_B, 'TRACK B')

    with open(f'{WORKING}/radiomics_trackA_results.json', 'w') as f:
        json.dump({'fold_results':  results_A,
                   'all_val_preds': preds_A}, f)
    with open(f'{WORKING}/radiomics_trackB_results.json', 'w') as f:
        json.dump({'fold_results':  results_B,
                   'all_val_preds': preds_B}, f)

    if shap_A:
        pd.DataFrame(shap_A).to_csv(
            f'{WORKING}/radiomics_trackA_shap.csv', index=False)
    if shap_B:
        pd.DataFrame(shap_B).to_csv(
            f'{WORKING}/radiomics_trackB_shap.csv', index=False)

    commit_to_kaggle(
        'Radiomics Cell 8 complete — SVM + XGBoost + SHAP')

    print(f'\n✅ Cell 8 complete')
    print(f'Next: Cell 9 — Hybrid Fusion embeddings')

## CELL 9: HYBRID FUSION — EMBEDDING EXTRACTION + FUSION

In [ ]:
# ── CELL 9: HYBRID FUSION — EMBEDDING EXTRACTION + FUSION ────────────────────
# Early fusion: CNN embeddings + radiomics features → SVM/XGBoost
# Late fusion:  average CNN prob + radiomics prob
#
# For each fold:
#   1. Load CNN best weights
#   2. Extract 512-dim embeddings (remove final layer)
#   3. Average across slices → one vector per patient
#   4. PCA compress to 50 dims (fit on train fold only)
#   5. Concatenate with selected radiomic features
#   6. Train SVM + XGBoost on combined vector
#   7. Late fusion: average probabilities
#
# GPU required for embedding extraction
# Time: ~45-60 minutes

import os, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    f1_score, confusion_matrix, brier_score_loss
)
import xgboost as xgb

warnings.filterwarnings('ignore')

WORKING    = '/kaggle/working'
SLICES_DIR = '/kaggle/input/datasets/adesaladaniel/glioma-idh-tensors'
MODELS_DATASET = 'adesaladaniel/glioma-idh-models'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FOLDS      = 5
CLASS_WEIGHT = 3.81
SEED         = 42
BATCH_SIZE   = 64
PCA_DIMS     = 50
META_COLS    = ['patient_id', 'IDH_binary', 'fold']

print(f'Device     : {DEVICE}')
print(f'PCA dims   : {PCA_DIMS}')
print(f'Slices dir : {SLICES_DIR}')

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df_labels = pd.read_csv(f'{WORKING}/labels.csv')
df_A      = pd.read_csv(f'{WORKING}/radiomics_trackA.csv')
df_B      = pd.read_csv(f'{WORKING}/radiomics_trackB.csv')

with open(f'{WORKING}/patient_splits.json') as f:
    splits = json.load(f)

with open(f'{WORKING}/radiomics_trackA_results.json') as f:
    rad_results_A = json.load(f)
with open(f'{WORKING}/radiomics_trackB_results.json') as f:
    rad_results_B = json.load(f)

label_map = dict(zip(df_labels['patient_id'],
                     df_labels['IDH_binary'].astype(int)))

feature_cols_A = [c for c in df_A.columns if c not in META_COLS]
feature_cols_B = [c for c in df_B.columns if c not in META_COLS]

print(f'Labels     : {len(df_labels)} patients')
print(f'Track A    : {len(feature_cols_A)} radiomic features')
print(f'Track B    : {len(feature_cols_B)} radiomic features')

# ── MODEL — EMBEDDING EXTRACTOR ───────────────────────────────────────────────
def build_embedding_model(n_channels, weights_path):
    """
    Load trained CNN, remove final classification layer.
    Returns model that outputs 512-dim embedding per slice.
    """
    model = resnet18(weights=None)

    new_conv = nn.Conv2d(n_channels, 64, kernel_size=7,
                         stride=2, padding=3, bias=False)
    model.conv1 = new_conv
    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )

    state = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(state)

    # Remove final classification layers — keep up to 512-dim output
    # Replace fc with identity to get 512-dim embedding
    model.fc = nn.Identity()

    model = model.to(DEVICE)
    model.eval()
    return model

# ── DATASET ───────────────────────────────────────────────────────────────────
class EmbeddingDataset(torch.utils.data.Dataset):
    def __init__(self, patient_ids, slices_dir, n_channels):
        self.slices_dir = slices_dir
        self.n_channels = n_channels
        self.items      = []
        for pid in patient_ids:
            npy_path = os.path.join(slices_dir, f'{pid}.npy')
            if not os.path.exists(npy_path):
                continue
            arr = np.load(npy_path, mmap_mode='r')
            for s in range(arr.shape[0]):
                self.items.append((pid, s))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        pid, s = self.items[idx]
        arr = np.load(
            os.path.join(self.slices_dir, f'{pid}.npy'),
            mmap_mode='r'
        )
        x = torch.tensor(
            arr[s, :self.n_channels].astype(np.float32),
            dtype=torch.float32
        )
        return x, pid

# ── EXTRACT EMBEDDINGS ────────────────────────────────────────────────────────
def extract_embeddings(patient_ids, model, slices_dir,
                       n_channels, device):
    """
    Run all patients through CNN backbone.
    Average slice embeddings → one 512-dim vector per patient.
    Returns dict: patient_id → 512-dim numpy array
    """
    dataset = EmbeddingDataset(patient_ids, slices_dir, n_channels)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2,
                         pin_memory=True)

    patient_embeddings = {}

    with torch.no_grad():
        for x, pids in loader:
            x    = x.to(device)
            embs = model(x)  # (batch, 512)
            embs = embs.cpu().numpy()
            for emb, pid in zip(embs, pids):
                if pid not in patient_embeddings:
                    patient_embeddings[pid] = []
                patient_embeddings[pid].append(emb)

    # Average across slices
    return {
        pid: np.mean(np.stack(embs), axis=0)
        for pid, embs in patient_embeddings.items()
    }

# ── METRICS ───────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)
    return {
        'patient_auc': round(float(roc_auc_score(y_true, y_prob)), 4),
        'accuracy':    round(float(accuracy_score(y_true, y_pred)), 4),
        'sensitivity': round(float(recall_score(
            y_true, y_pred, zero_division=0)), 4),
        'specificity': round(float(
            tn/(tn+fp) if (tn+fp) > 0 else 0.0), 4),
        'f1':          round(float(f1_score(
            y_true, y_pred, zero_division=0)), 4),
        'brier':       round(float(
            brier_score_loss(y_true, y_prob)), 4),
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }

# ── FEATURE SELECTION (same as Cell 8) ───────────────────────────────────────
def select_features(X_train, y_train, feature_names):
    selected = list(feature_names)
    X_work   = X_train.copy()

    vt = VarianceThreshold(threshold=0.01)
    vt.fit(X_work)
    mask_vt    = vt.get_support()
    X_work     = X_work[:, mask_vt]
    selected   = [n for n, m in zip(selected, mask_vt) if m]

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_work)

    lasso = LassoCV(cv=3, max_iter=5000,
                    random_state=SEED, n_jobs=-1)
    lasso.fit(X_scaled, y_train)
    mask_lasso = np.abs(lasso.coef_) > 1e-6
    selected   = [n for n, m in zip(selected, mask_lasso) if m]

    return selected, vt, scaler, mask_lasso

# ── COMMIT TO KAGGLE ──────────────────────────────────────────────────────────
def commit_to_kaggle(message):
    commit_dir = f'{WORKING}/fusion_commit'
    os.makedirs(commit_dir, exist_ok=True)

    with open(os.path.join(commit_dir,
                           'dataset-metadata.json'), 'w') as f:
        json.dump({'title':     'Glioma IDH Models',
                   'id':        MODELS_DATASET,
                   'licenses':  [{'name': 'other'}],
                   'isPrivate': True}, f)

    for fname in os.listdir(WORKING):
        src = os.path.join(WORKING, fname)
        lnk = os.path.join(commit_dir, fname)
        if any(fname.startswith(p) for p in
               ['cnn_track', 'radiomics_', 'fusion_']) and \
           any(fname.endswith(e) for e in
               ['.pth', '.json', '.csv']):
            if os.path.exists(src) and not os.path.lexists(lnk):
                os.symlink(os.path.abspath(src), lnk)

    r = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', commit_dir, '-m', message,
         '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
    status = '✅' if r.returncode == 0 else '⚠️'
    print(f'  {status} Kaggle: {message}')
    shutil.rmtree(commit_dir, ignore_errors=True)

# ── MAIN FUSION FUNCTION ──────────────────────────────────────────────────────
def run_fusion(track_name, n_channels, df_rad,
               feature_cols, cnn_progress_path):
    """
    Full hybrid fusion pipeline for one track.
    """
    print(f'\n{"="*60}')
    print(f'HYBRID FUSION — {track_name}')
    print(f'{"="*60}')

    # Load CNN OOF predictions (for late fusion)
    with open(cnn_progress_path) as f:
        cnn_progress = json.load(f)
    cnn_oof = cnn_progress.get('all_val_preds', {})

    # Load radiomics OOF predictions (for late fusion)
    rad_key = 'A' if 'A' in track_name else 'B'
    rad_results = (rad_results_A if rad_key == 'A'
                   else rad_results_B)
    rad_oof = rad_results.get('all_val_preds', {})

    fold_results  = {}
    all_val_preds = {}

    for fold in range(N_FOLDS):
        print(f'\nFold {fold}')
        print('-'*40)

        val_patients   = splits[f'fold_{fold}']
        train_patients = [p for f2 in range(N_FOLDS)
                          if f2 != fold
                          for p in splits[f'fold_{f2}']]

        # ── LOAD CNN WEIGHTS ──────────────────────────────────────
        weights_path = (f'{WORKING}/cnn_track'
                        f'{"A" if "A" in track_name else "B"}'
                        f'_fold{fold}_best.pth')

        if not os.path.exists(weights_path):
            print(f'  ❌ Weights not found: {weights_path}')
            continue

        model = build_embedding_model(n_channels, weights_path)
        print(f'  CNN weights loaded ✅')

        # ── EXTRACT EMBEDDINGS ────────────────────────────────────
        print(f'  Extracting embeddings...')

        train_embs = extract_embeddings(
            train_patients, model, SLICES_DIR,
            n_channels, DEVICE)
        val_embs = extract_embeddings(
            val_patients, model, SLICES_DIR,
            n_channels, DEVICE)

        print(f'  Train embeddings: {len(train_embs)} patients '
              f'× 512 dims')
        print(f'  Val embeddings  : {len(val_embs)} patients '
              f'× 512 dims')

        # Free GPU memory
        del model
        torch.cuda.empty_cache()

        # ── PCA ON EMBEDDINGS (fit on train only) ─────────────────
        train_pids = [p for p in train_patients
                      if p in train_embs]
        val_pids   = [p for p in val_patients
                      if p in val_embs]

        X_emb_train = np.stack(
            [train_embs[p] for p in train_pids])
        X_emb_val   = np.stack(
            [val_embs[p] for p in val_pids])

        y_train = np.array([label_map[p] for p in train_pids])
        y_val   = np.array([label_map[p] for p in val_pids])

        pca = PCA(n_components=min(PCA_DIMS,
                                   X_emb_train.shape[1],
                                   X_emb_train.shape[0]-1),
                  random_state=SEED)
        X_pca_train = pca.fit_transform(X_emb_train)
        X_pca_val   = pca.transform(X_emb_val)

        print(f'  PCA: 512 → {X_pca_train.shape[1]} dims '
              f'({pca.explained_variance_ratio_.sum()*100:.1f}% '
              f'variance)')

        # ── RADIOMICS FEATURES ────────────────────────────────────
        rad_train = df_rad[
            df_rad['patient_id'].isin(train_pids)
        ].set_index('patient_id')
        rad_val = df_rad[
            df_rad['patient_id'].isin(val_pids)
        ].set_index('patient_id')

        X_rad_train = rad_train.loc[
            train_pids, feature_cols].values.astype(np.float32)
        X_rad_val   = rad_val.loc[
            val_pids, feature_cols].values.astype(np.float32)

        # Feature selection on radiomics (train only)
        selected, vt, scaler, mask_lasso = select_features(
            X_rad_train, y_train, feature_cols)

        mask_vt = vt.get_support()
        X_rad_tr_vt = X_rad_train[:, mask_vt]
        X_rad_vl_vt = X_rad_val[:, mask_vt]

        X_rad_tr_sc = scaler.transform(X_rad_tr_vt)
        X_rad_vl_sc = scaler.transform(X_rad_vl_vt)

        all_after_vt = [f for f, m in zip(feature_cols, mask_vt)
                        if m]
        lasso_idx = [i for i, n in enumerate(all_after_vt)
                     if n in selected]

        X_rad_tr_sel = X_rad_tr_sc[:, lasso_idx]
        X_rad_vl_sel = X_rad_vl_sc[:, lasso_idx]

        print(f'  Radiomics: {len(selected)} features selected')

        # ── EARLY FUSION — CONCATENATE ────────────────────────────
        X_fused_train = np.concatenate(
            [X_pca_train, X_rad_tr_sel], axis=1)
        X_fused_val   = np.concatenate(
            [X_pca_val, X_rad_vl_sel], axis=1)

        print(f'  Fused vector: {X_fused_train.shape[1]} dims '
              f'({X_pca_train.shape[1]} PCA + '
              f'{X_rad_tr_sel.shape[1]} radiomics)')

        # ── EARLY FUSION SVM ──────────────────────────────────────
        svm = SVC(kernel='rbf', probability=True,
                  class_weight={0: 1.0, 1: CLASS_WEIGHT},
                  C=1.0, gamma='scale', random_state=SEED)
        svm.fit(X_fused_train, y_train)
        svm_probs = svm.predict_proba(X_fused_val)[:, 1]
        svm_preds = (svm_probs >= 0.5).astype(int)
        svm_m     = compute_metrics(y_val, svm_preds, svm_probs)

        # ── EARLY FUSION XGBOOST ──────────────────────────────────
        scale_pos = (y_train==0).sum() / (y_train==1).sum()
        xgb_model = xgb.XGBClassifier(
            n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos,
            eval_metric='logloss',
            random_state=SEED, n_jobs=-1
        )
        xgb_model.fit(X_fused_train, y_train,
                      eval_set=[(X_fused_val, y_val)],
                      verbose=False)
        xgb_probs = xgb_model.predict_proba(X_fused_val)[:, 1]
        xgb_preds = (xgb_probs >= 0.5).astype(int)
        xgb_m     = compute_metrics(y_val, xgb_preds, xgb_probs)

        # ── LATE FUSION ───────────────────────────────────────────
        late_probs = []
        late_labels = []
        late_pids   = []

        for pid in val_pids:
            if pid in cnn_oof and pid in rad_oof:
                cnn_p = cnn_oof[pid]['mean_prob']
                rad_p = rad_oof[pid]['mean_prob']
                avg_p = (cnn_p + rad_p) / 2.0
                late_probs.append(avg_p)
                late_labels.append(label_map[pid])
                late_pids.append(pid)

        late_preds = [1 if p >= 0.5 else 0
                      for p in late_probs]
        late_m = compute_metrics(
            late_labels, late_preds, late_probs) \
            if late_probs else {}

        # ── PRINT FOLD RESULTS ────────────────────────────────────
        print(f'\n  Early Fusion SVM    : '
              f'AUC {svm_m["patient_auc"]:.4f} | '
              f'Sens {svm_m["sensitivity"]:.4f} | '
              f'Spec {svm_m["specificity"]:.4f} | '
              f'F1 {svm_m["f1"]:.4f}')
        print(f'  Early Fusion XGBoost: '
              f'AUC {xgb_m["patient_auc"]:.4f} | '
              f'Sens {xgb_m["sensitivity"]:.4f} | '
              f'Spec {xgb_m["specificity"]:.4f} | '
              f'F1 {xgb_m["f1"]:.4f}')
        if late_m:
            print(f'  Late Fusion (avg)   : '
                  f'AUC {late_m["patient_auc"]:.4f} | '
                  f'Sens {late_m["sensitivity"]:.4f} | '
                  f'Spec {late_m["specificity"]:.4f} | '
                  f'F1 {late_m["f1"]:.4f}')

        # Best early fusion
        if xgb_m['patient_auc'] >= svm_m['patient_auc']:
            best_probs = xgb_probs
            best_name  = 'XGBoost'
        else:
            best_probs = svm_probs
            best_name  = 'SVM'

        # Store OOF
        for pid, prob in zip(val_pids, best_probs):
            all_val_preds[pid] = {
                'mean_prob': float(prob),
                'label':     int(label_map[pid]),
                'fold':      fold
            }

        fold_results[str(fold)] = {
            'early_svm':     svm_m,
            'early_xgb':     xgb_m,
            'late_fusion':   late_m,
            'best_early':    best_name,
            'n_pca_dims':    int(X_pca_train.shape[1]),
            'n_rad_features': len(selected),
            'n_fused_dims':  int(X_fused_train.shape[1])
        }

    # ── FINAL SUMMARY ─────────────────────────────────────────────────────────
    print(f'\n{"="*60}')
    print(f'FUSION {track_name} — SUMMARY')
    print(f'{"="*60}')

    early_svm_aucs = [
        fold_results[str(f)]['early_svm']['patient_auc']
        for f in range(N_FOLDS) if str(f) in fold_results]
    early_xgb_aucs = [
        fold_results[str(f)]['early_xgb']['patient_auc']
        for f in range(N_FOLDS) if str(f) in fold_results]
    late_aucs = [
        fold_results[str(f)]['late_fusion']['patient_auc']
        for f in range(N_FOLDS)
        if str(f) in fold_results and
        fold_results[str(f)]['late_fusion']]

    print(f'\nEarly Fusion SVM    : '
          f'{np.mean(early_svm_aucs):.4f} ± '
          f'{np.std(early_svm_aucs):.4f}')
    print(f'Early Fusion XGBoost: '
          f'{np.mean(early_xgb_aucs):.4f} ± '
          f'{np.std(early_xgb_aucs):.4f}')
    if late_aucs:
        print(f'Late Fusion (avg)   : '
              f'{np.mean(late_aucs):.4f} ± '
              f'{np.std(late_aucs):.4f}')

    return fold_results, all_val_preds

# ── RUN TRACK A FUSION ────────────────────────────────────────────────────────
results_fusion_A, preds_fusion_A = run_fusion(
    track_name        = 'TRACK A',
    n_channels        = 7,
    df_rad            = df_A,
    feature_cols      = feature_cols_A,
    cnn_progress_path = f'{WORKING}/cnn_trackA_progress.json'
)

# ── RUN TRACK B FUSION ────────────────────────────────────────────────────────
results_fusion_B, preds_fusion_B = run_fusion(
    track_name        = 'TRACK B',
    n_channels        = 4,
    df_rad            = df_B,
    feature_cols      = feature_cols_B,
    cnn_progress_path = f'{WORKING}/cnn_trackB_progress.json'
)

# ── SAVE RESULTS ──────────────────────────────────────────────────────────────
print(f'\nSaving fusion results...')

with open(f'{WORKING}/fusion_trackA_results.json', 'w') as f:
    json.dump({'fold_results':  results_fusion_A,
               'all_val_preds': preds_fusion_A}, f)

with open(f'{WORKING}/fusion_trackB_results.json', 'w') as f:
    json.dump({'fold_results':  results_fusion_B,
               'all_val_preds': preds_fusion_B}, f)

commit_to_kaggle('Fusion Cell 9 complete — Track A + Track B')

# ── GRAND SUMMARY ─────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('COMPLETE RESULTS — ALL ARMS')
print(f'{"="*60}')

def best_fusion_auc(results):
    svm = [results[str(f)]['early_svm']['patient_auc']
           for f in range(5) if str(f) in results]
    xgb = [results[str(f)]['early_xgb']['patient_auc']
           for f in range(5) if str(f) in results]
    late = [results[str(f)]['late_fusion']['patient_auc']
            for f in range(5)
            if str(f) in results and
            results[str(f)]['late_fusion']]
    best_early = max(np.mean(svm), np.mean(xgb))
    best_early_std = (np.std(svm)
                      if np.mean(svm) >= np.mean(xgb)
                      else np.std(xgb))
    late_mean = np.mean(late) if late else 0
    late_std  = np.std(late)  if late else 0
    return best_early, best_early_std, late_mean, late_std

eA, eA_s, lA, lA_s = best_fusion_auc(results_fusion_A)
eB, eB_s, lB, lB_s = best_fusion_auc(results_fusion_B)

print(f'\n{"Model":<35} {"Track A":>10} {"Track B":>10}')
print('─'*57)
print(f'{"CNN (ResNet18)":<35} '
      f'{"0.9654":>10} {"0.9636":>10}')
print(f'{"Radiomics (best)":<35} '
      f'{"0.9642":>10} {"0.9722":>10}')
print(f'{"Early Fusion (best)":<35} '
      f'{eA:>10.4f} {eB:>10.4f}')
print(f'{"Late Fusion":<35} '
      f'{lA:>10.4f} {lB:>10.4f}')
print()
print(f'Literature benchmark      : ~0.90')
print(f'2023 Frontiers paper      : ~0.90')
print()
print(f'✅ Cell 9 complete')
print(f'Next: Cell 10 — External validation (EGD + TCGA)')